In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

historical_clean = pd.read_pickle(
    "historical_player_match_prepared.pkl"
)

print("Shape:", historical_clean.shape)
print("Matches:", historical_clean["match_id"].nunique())
print("Players:", historical_clean["player_name"].nunique())

historical_clean.head()

Shape: (113470, 21)
Matches: 3961
Players: 9908


,match_id,player_name,total_events,pass_attempts,passes_completed,forward_passes,progressive_passes,avg_pass_length,avg_pass_angle_degrees,crosses,...,switches,shot_assists,goal_assists,shots,carries,miscontrols,pass_completion_rate,match_date,home_team_name,away_team_name
0,15946,Adrián Marín Gómez,29,7,5,5,2,16.931792,47.062771,0,...,0,0,0,1,5,1,71.428571,2018-08-18,Barcelona,Deportivo Alavés
1,15946,Arthur Henrique Ramos de Oliveira Melo,53,18,17,10,1,20.407317,14.809665,0,...,1,0,1,0,15,0,94.444444,2018-08-18,Barcelona,Deportivo Alavés
2,15946,Arturo Erasmo Vidal Pardo,22,7,7,1,0,10.627395,40.042062,0,...,0,0,0,0,6,0,100.000000,2018-08-18,Barcelona,Deportivo Alavés
3,15946,Borja González Tomás,23,6,4,3,0,11.737756,-57.250195,0,...,0,0,0,0,3,0,66.666667,2018-08-18,Barcelona,Deportivo Alavés
4,15946,Daniel Alejandro Torres Rojas,56,16,12,12,1,15.537846,4.651870,0,...,0,0,0,0,9,0,75.000000,2018-08-18,Barcelona,Deportivo Alavés


In [2]:
historical_clean = historical_clean.sort_values(
    by=["player_name", "match_date", "match_id"]
).reset_index(drop=True)

historical_clean[
    [
        "player_name",
        "match_date",
        "match_id",
        "pass_attempts",
        "pass_completion_rate",
        "progressive_passes",
        "carries"
    ]
].head(20)

,player_name,match_date,match_id,pass_attempts,pass_completion_rate,progressive_passes,carries
0,Aaren D''Silva,2021-12-08,3813299,4,50.000000,0,1
1,Aaren D''Silva,2021-12-13,3813290,2,50.000000,1,1
2,Aaren D''Silva,2021-12-18,3813272,3,66.666667,1,4
3,Aaren D''Silva,2021-12-23,3813304,9,55.555556,1,13
4,Aaren D''Silva,2022-01-13,3817879,3,0.000000,0,3
5,Aaren D''Silva,2022-02-11,3817869,1,100.000000,0,1
6,Aaren D''Silva,2022-03-12,3827338,3,66.666667,1,3
7,Aaren D''Silva,2022-03-16,3827336,8,75.000000,2,6
8,Aaren D''Silva,2022-03-20,3827767,10,70.000000,3,10
9,Aaron Cresswell,2015-08-09,3754141,48,68.750000,11,27


In [3]:

historical_clean["prev_pass_completion_rate"] = (
    historical_clean
    .groupby("player_name")["pass_completion_rate"]
    .shift(1)
)

In [4]:
historical_clean[
    [
        "player_name",
        "match_date",
        "pass_completion_rate",
        "prev_pass_completion_rate"
    ]
].head(20)

,player_name,match_date,pass_completion_rate,prev_pass_completion_rate
0,Aaren D''Silva,2021-12-08,50.000000,NaN
1,Aaren D''Silva,2021-12-13,50.000000,50.000000
2,Aaren D''Silva,2021-12-18,66.666667,50.000000
3,Aaren D''Silva,2021-12-23,55.555556,66.666667
4,Aaren D''Silva,2022-01-13,0.000000,55.555556
5,Aaren D''Silva,2022-02-11,100.000000,0.000000
6,Aaren D''Silva,2022-03-12,66.666667,100.000000
7,Aaren D''Silva,2022-03-16,75.000000,66.666667
8,Aaren D''Silva,2022-03-20,70.000000,75.000000
9,Aaron Cresswell,2015-08-09,68.750000,NaN


In [5]:
validation_results = {
    "duplicate_player_match_rows": historical_clean.duplicated(
        subset=["match_id", "player_name"]
    ).sum(),

    "completed_greater_than_attempted": (
        historical_clean["passes_completed"]
        > historical_clean["pass_attempts"]
    ).sum(),

    "forward_greater_than_attempted": (
        historical_clean["forward_passes"]
        > historical_clean["pass_attempts"]
    ).sum(),

    "progressive_greater_than_attempted": (
        historical_clean["progressive_passes"]
        > historical_clean["pass_attempts"]
    ).sum(),

    "completion_above_100": (
        historical_clean["pass_completion_rate"] > 100
    ).sum(),

    "completion_below_0": (
        historical_clean["pass_completion_rate"] < 0
    ).sum(),

    "negative_pass_attempts": (
        historical_clean["pass_attempts"] < 0
    ).sum(),

    "missing_match_dates": (
        historical_clean["match_date"].isna().sum()
    )
}

for check, result in validation_results.items():
    print(f"{check}: {result}")

duplicate_player_match_rows: 0
completed_greater_than_attempted: 0
forward_greater_than_attempted: 0
progressive_greater_than_attempted: 0
completion_above_100: 0
completion_below_0: 0
negative_pass_attempts: 0
missing_match_dates: 0


In [11]:
historical_clean["rolling_3_pass_completion"] = (
    historical_clean
    .groupby("player_name")["pass_completion_rate"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

In [12]:
historical_clean["rolling_5_pass_completion"] = (
    historical_clean
    .groupby("player_name")["pass_completion_rate"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=5,
            min_periods=1
        ).mean()
    )
)

In [13]:
print(
    [
        col for col in historical_clean.columns
        if "rolling" in col
    ]
)

['rolling_5_pass_completion', 'rolling_3_pass_completion']


In [14]:
historical_clean[
    [
        "player_name",
        "match_date",
        "pass_completion_rate",
        "prev_pass_completion_rate",
        "rolling_3_pass_completion",
        "rolling_5_pass_completion"
    ]
].head(15)

,player_name,match_date,pass_completion_rate,prev_pass_completion_rate,rolling_3_pass_completion,rolling_5_pass_completion
0,Aaren D''Silva,2021-12-08,50.000000,NaN,NaN,NaN
1,Aaren D''Silva,2021-12-13,50.000000,50.000000,50.000000,50.000000
2,Aaren D''Silva,2021-12-18,66.666667,50.000000,50.000000,50.000000
3,Aaren D''Silva,2021-12-23,55.555556,66.666667,55.555556,55.555556
4,Aaren D''Silva,2022-01-13,0.000000,55.555556,57.407407,55.555556
5,Aaren D''Silva,2022-02-11,100.000000,0.000000,40.740741,44.444444
6,Aaren D''Silva,2022-03-12,66.666667,100.000000,51.851852,54.444444
7,Aaren D''Silva,2022-03-16,75.000000,66.666667,55.555556,57.777778
8,Aaren D''Silva,2022-03-20,70.000000,75.000000,80.555556,59.444444
9,Aaron Cresswell,2015-08-09,68.750000,NaN,NaN,NaN


In [15]:
historical_features = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for feature in historical_features:

    historical_clean[f"prev_{feature}"] = (
        historical_clean
        .groupby("player_name")[feature]
        .shift(1)
    )

    historical_clean[f"rolling_3_{feature}"] = (
        historical_clean
        .groupby("player_name")[feature]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )

    historical_clean[f"rolling_5_{feature}"] = (
        historical_clean
        .groupby("player_name")[feature]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )

In [16]:
historical_clean[
    [
        "player_name",
        "match_date",

        "progressive_passes",
        "prev_progressive_passes",
        "rolling_3_progressive_passes",
        "rolling_5_progressive_passes",

        "carries",
        "rolling_3_carries",
        "rolling_5_carries"
    ]
].head(15)

,player_name,match_date,progressive_passes,prev_progressive_passes,rolling_3_progressive_passes,rolling_5_progressive_passes,carries,rolling_3_carries,rolling_5_carries
0,Aaren D''Silva,2021-12-08,0,NaN,NaN,NaN,1,NaN,NaN
1,Aaren D''Silva,2021-12-13,1,0.0,0.000000,0.000000,1,1.000000,1.000000
2,Aaren D''Silva,2021-12-18,1,1.0,0.500000,0.500000,4,1.000000,1.000000
3,Aaren D''Silva,2021-12-23,1,1.0,0.666667,0.666667,13,2.000000,2.000000
4,Aaren D''Silva,2022-01-13,0,1.0,1.000000,0.750000,3,6.000000,4.750000
5,Aaren D''Silva,2022-02-11,0,0.0,0.666667,0.600000,1,6.666667,4.400000
6,Aaren D''Silva,2022-03-12,1,0.0,0.333333,0.600000,3,5.666667,4.400000
7,Aaren D''Silva,2022-03-16,2,1.0,0.333333,0.600000,6,2.333333,4.800000
8,Aaren D''Silva,2022-03-20,3,2.0,1.000000,0.800000,10,3.333333,5.200000
9,Aaron Cresswell,2015-08-09,11,NaN,NaN,NaN,27,NaN,NaN


In [17]:
historical_clean["previous_matches_available"] = (
    historical_clean
    .groupby("player_name")
    .cumcount()
)

In [18]:
historical_clean[
    [
        "player_name",
        "match_date",
        "previous_matches_available",
        "rolling_3_pass_completion",
        "rolling_5_pass_completion"
    ]
].head(20)

,player_name,match_date,previous_matches_available,rolling_3_pass_completion,rolling_5_pass_completion
0,Aaren D''Silva,2021-12-08,0,NaN,NaN
1,Aaren D''Silva,2021-12-13,1,50.000000,50.000000
2,Aaren D''Silva,2021-12-18,2,50.000000,50.000000
3,Aaren D''Silva,2021-12-23,3,55.555556,55.555556
4,Aaren D''Silva,2022-01-13,4,57.407407,55.555556
5,Aaren D''Silva,2022-02-11,5,40.740741,44.444444
6,Aaren D''Silva,2022-03-12,6,51.851852,54.444444
7,Aaren D''Silva,2022-03-16,7,55.555556,57.777778
8,Aaren D''Silva,2022-03-20,8,80.555556,59.444444
9,Aaron Cresswell,2015-08-09,0,NaN,NaN


In [19]:
historical_clean["previous_matches_available"] = (
    historical_clean
    .groupby("player_name")
    .cumcount()
)

In [20]:
historical_clean[
    [
        "player_name",
        "match_date",
        "previous_matches_available",
        "rolling_3_pass_completion",
        "rolling_5_pass_completion"
    ]
].head(20)

,player_name,match_date,previous_matches_available,rolling_3_pass_completion,rolling_5_pass_completion
0,Aaren D''Silva,2021-12-08,0,NaN,NaN
1,Aaren D''Silva,2021-12-13,1,50.000000,50.000000
2,Aaren D''Silva,2021-12-18,2,50.000000,50.000000
3,Aaren D''Silva,2021-12-23,3,55.555556,55.555556
4,Aaren D''Silva,2022-01-13,4,57.407407,55.555556
5,Aaren D''Silva,2022-02-11,5,40.740741,44.444444
6,Aaren D''Silva,2022-03-12,6,51.851852,54.444444
7,Aaren D''Silva,2022-03-16,7,55.555556,57.777778
8,Aaren D''Silva,2022-03-20,8,80.555556,59.444444
9,Aaron Cresswell,2015-08-09,0,NaN,NaN


In [21]:
import json
from pathlib import Path

lineup_file = Path(
    "open-data-master/data/lineups/3764440.json"
)

with open(lineup_file, "r", encoding="utf-8") as f:
    lineup_data = json.load(f)

print(type(lineup_data))
print("Number of teams:", len(lineup_data))

<class 'list'>
Number of teams: 2


In [22]:
lineup_data[0]

{'team_id': 1042,
 'team_name': 'Elche',
 'lineup': [{'player_id': 3246,
   'player_name': 'Guido Marcelo Carrillo',
   'player_nickname': 'Guido Carrillo',
   'jersey_number': 21,
   'country': {'id': 11, 'name': 'Argentina'},
   'cards': [],
   'positions': [{'position_id': 23,
     'position': 'Center Forward',
     'from': '71:23',
     'to': None,
     'from_period': 2,
     'to_period': None,
     'start_reason': 'Substitution - On (Tactical)',
     'end_reason': 'Final Whistle'}]},
  {'player_id': 5109,
   'player_name': 'Paulo Dino Gazzaniga',
   'player_nickname': 'Paulo Gazzaniga',
   'jersey_number': 1,
   'country': {'id': 11, 'name': 'Argentina'},
   'cards': [],
   'positions': []},
  {'player_id': 5691,
   'player_name': 'Johan Andrés Mojica Palacio',
   'player_nickname': 'Johan Mojica',
   'jersey_number': 25,
   'country': {'id': 49, 'name': 'Colombia'},
   'cards': [],
   'positions': [{'position_id': 8,
     'position': 'Left Wing Back',
     'from': '00:00',
     '

In [23]:
lineup_summary = []

for team in lineup_data:

    team_name = team["team_name"]

    for player in team["lineup"]:

        positions = player.get("positions", [])

        if len(positions) > 0:
            first_position = positions[0]

            lineup_summary.append({
                "team_name": team_name,
                "player_name": player["player_name"],
                "position": first_position.get("position"),
                "from": first_position.get("from"),
                "to": first_position.get("to"),
                "start_reason": first_position.get("start_reason"),
                "end_reason": first_position.get("end_reason")
            })

lineup_summary_df = pd.DataFrame(lineup_summary)

lineup_summary_df

,team_name,player_name,position,from,to,start_reason,end_reason
0,Elche,Guido Marcelo Carrillo,Center Forward,71:23,NaN,Substitution - On (Tactical),Final Whistle
1,Elche,Johan Andrés Mojica Palacio,Left Wing Back,00:00,37:02,Starting XI,Player Off
2,Elche,Antonio Barragán Fernández,Right Center Back,00:00,NaN,Starting XI,Final Whistle
3,Elche,Lucas Ariel Boyé,Center Forward,00:00,71:23,Starting XI,Substitution - Off (Tactical)
4,Elche,Miguel Ángel Garrido Cifuentes,Right Wing Back,00:00,59:20,Starting XI,Substitution - Off (Tactical)
5,Elche,José Manuel Sánchez Guillén,Left Center Back,00:00,NaN,Starting XI,Final Whistle
6,Elche,Pere Milla Peña,Right Wing,00:00,71:37,Starting XI,Substitution - Off (Tactical)
7,Elche,Emiliano Ariel Rigoni,Left Wing,00:00,59:25,Starting XI,Substitution - Off (Tactical)
8,Elche,Omenuke Mfulu,Left Center Midfield,00:00,80:12,Starting XI,Substitution - Off (Injury)
9,Elche,Fidel Chaves de la Torre,Right Wing Back,59:20,65:03,Substitution - On (Tactical),Tactical Shift


In [24]:
messi_positions = None

for team in lineup_data:
    for player in team["lineup"]:
        if player["player_name"] == "Lionel Andrés Messi Cuccittini":
            messi_positions = player["positions"]

messi_positions

[{'position_id': 23,
  'position': 'Center Forward',
  'from': '00:00',
  'to': '45:00',
  'from_period': 1,
  'to_period': 2,
  'start_reason': 'Starting XI',
  'end_reason': 'Tactical Shift'},
 {'position_id': 19,
  'position': 'Center Attacking Midfield',
  'from': '45:00',
  'to': None,
  'from_period': 2,
  'to_period': None,
  'start_reason': 'Tactical Shift',
  'end_reason': 'Final Whistle'}]

In [25]:
fidel_positions = None

for team in lineup_data:
    for player in team["lineup"]:
        if player["player_name"] == "Fidel Chaves de la Torre":
            fidel_positions = player["positions"]

fidel_positions

[{'position_id': 7,
  'position': 'Right Wing Back',
  'from': '59:20',
  'to': '65:03',
  'from_period': 2,
  'to_period': 2,
  'start_reason': 'Substitution - On (Tactical)',
  'end_reason': 'Tactical Shift'},
 {'position_id': 21,
  'position': 'Left Wing',
  'from': '65:03',
  'to': None,
  'from_period': 2,
  'to_period': None,
  'start_reason': 'Tactical Shift',
  'end_reason': 'Final Whistle'}]

In [26]:
fidel_positions = None

for team in lineup_data:
    for player in team["lineup"]:
        if player["player_name"] == "Fidel Chaves de la Torre":
            fidel_positions = player["positions"]

fidel_positions

[{'position_id': 7,
  'position': 'Right Wing Back',
  'from': '59:20',
  'to': '65:03',
  'from_period': 2,
  'to_period': 2,
  'start_reason': 'Substitution - On (Tactical)',
  'end_reason': 'Tactical Shift'},
 {'position_id': 21,
  'position': 'Left Wing',
  'from': '65:03',
  'to': None,
  'from_period': 2,
  'to_period': None,
  'start_reason': 'Tactical Shift',
  'end_reason': 'Final Whistle'}]

In [28]:
events_file = Path(
    "open-data-master/data/events/3764440.json"
)

with open(events_file, "r", encoding="utf-8") as f:
    events_3764440 = json.load(f)

events_test = pd.DataFrame(events_3764440)

events_test["event_type"] = events_test["type"].apply(
    lambda x: x.get("name")
    if isinstance(x, dict)
    else x
)

events_test[
    [
        "period",
        "timestamp",
        "minute",
        "second",
        "event_type"
    ]
].tail(20)

,period,timestamp,minute,second,event_type
4140,2,00:47:43.760,92,43,Pressure
4141,2,00:47:44.668,92,44,Pass
4142,2,00:47:45.088,92,45,Pressure
4143,2,00:47:45.729,92,45,Ball Receipt*
4144,2,00:47:45.729,92,45,Carry
4145,2,00:47:46.346,92,46,Dispossessed
4146,2,00:47:46.346,92,46,Duel
4147,2,00:47:46.896,92,46,Pressure
4148,2,00:47:46.922,92,46,Pressure
4149,2,00:47:47.782,92,47,Block


In [30]:
lineups_path = Path(
    "open-data-master/data/lineups"
)

lineup_files = list(
    lineups_path.glob("*.json")
)

print(
    "Lineup files found:",
    len(lineup_files)
)

Lineup files found: 4235


In [31]:
sub_events = events_test[
    events_test["event_type"] == "Substitution"
].copy()

sub_events[
    [
        "minute",
        "second",
        "timestamp",
        "player",
        "substitution"
    ]
]

,minute,second,timestamp,player,substitution
1997,45,0,00:00:00.000,"{'id': 6947, 'name': 'Miralem Pjanić'}","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
2744,59,20,00:14:20.006,"{'id': 7897, 'name': 'Miguel Ángel Garrido Cif...","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
2745,59,25,00:14:25.731,"{'id': 15880, 'name': 'Emiliano Ariel Rigoni'}","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3007,65,49,00:20:49.545,"{'id': 22390, 'name': 'Francisco António Macha...","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3228,71,23,00:26:23.815,"{'id': 7064, 'name': 'Lucas Ariel Boyé'}","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3229,71,37,00:26:37.094,"{'id': 12072, 'name': 'Pere Milla Peña'}","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3367,75,18,00:30:18.344,"{'id': 4447, 'name': 'Martin Braithwaite Chris...","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3368,75,33,00:30:33.563,"{'id': 5213, 'name': 'Gerard Piqué Bernabéu'}","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3370,76,8,00:31:08.986,"{'id': 5211, 'name': 'Jordi Alba Ramos'}","{'outcome': {'id': 103, 'name': 'Tactical'}, '..."
3586,80,12,00:35:12.799,"{'id': 18783, 'name': 'Omenuke Mfulu'}","{'outcome': {'id': 102, 'name': 'Injury'}, 're..."


In [32]:
def clock_to_seconds(clock_value):

    if pd.isna(clock_value):
        return np.nan

    minutes, seconds = clock_value.split(":")

    return (
        int(minutes) * 60
        + float(seconds)
    )

In [33]:
print("00:00 =", clock_to_seconds("00:00"))
print("45:00 =", clock_to_seconds("45:00"))
print("59:20 =", clock_to_seconds("59:20"))
print("71:23 =", clock_to_seconds("71:23"))
print("92:59 =", clock_to_seconds("92:59"))

00:00 = 0.0
45:00 = 2700.0
59:20 = 3560.0
71:23 = 4283.0
92:59 = 5579.0


In [34]:
match_end_seconds = (
    events_test["minute"].max() * 60
    + events_test.loc[
        events_test["minute"] == events_test["minute"].max(),
        "second"
    ].max()
)

match_end_minutes = match_end_seconds / 60

print("Match end seconds:", match_end_seconds)
print("Match duration:", match_end_minutes)

Match end seconds: 5579
Match duration: 92.98333333333333


In [35]:
def calculate_player_minutes(lineup_data, match_end_seconds):

    player_minutes = []

    for team in lineup_data:

        team_name = team["team_name"]

        for player in team["lineup"]:

            positions = player.get("positions", [])

            # Player was listed but never entered the pitch
            if len(positions) == 0:

                player_minutes.append({
                    "team_name": team_name,
                    "player_name": player["player_name"],
                    "minutes_played": 0.0,
                    "position_changes": 0
                })

                continue

            # First time the player entered the pitch
            start_time = positions[0].get("from")

            start_seconds = clock_to_seconds(start_time)

            # Final position record tells us how the
            # player's appearance ended
            final_position = positions[-1]

            end_time = final_position.get("to")

            if pd.isna(end_time):
                end_seconds = match_end_seconds
            else:
                end_seconds = clock_to_seconds(end_time)

            minutes_played = (
                end_seconds - start_seconds
            ) / 60

            # More than one position record means
            # at least one recorded position change
            position_changes = max(
                len(positions) - 1,
                0
            )

            player_minutes.append({
                "team_name": team_name,
                "player_name": player["player_name"],
                "minutes_played": minutes_played,
                "position_changes": position_changes
            })

    return pd.DataFrame(player_minutes)

In [36]:
minutes_test = calculate_player_minutes(
    lineup_data,
    match_end_seconds
)

minutes_test.sort_values(
    ["team_name", "minutes_played"],
    ascending=[True, False]
)

,team_name,player_name,minutes_played,position_changes
29,Barcelona,Samuel Yves Umtiti,92.983333,1
30,Barcelona,Lionel Andrés Messi Cuccittini,92.983333,1
34,Barcelona,Frenkie de Jong,92.983333,1
36,Barcelona,Marc-André ter Stegen,92.983333,0
41,Barcelona,Pedro González López,92.983333,1
44,Barcelona,Óscar Mingueza García,92.983333,0
25,Barcelona,Jordi Alba Ramos,76.133333,0
26,Barcelona,Gerard Piqué Bernabéu,75.550000,0
23,Barcelona,Martin Braithwaite Christensen,75.300000,1
38,Barcelona,Francisco António Machado Mota de Castro Trincão,65.816667,0


In [37]:
players_to_check = [
    "Lionel Andrés Messi Cuccittini",
    "Fidel Chaves de la Torre",
    "Jordi Alba Ramos",
    "Miralem Pjanić",
    "Guido Marcelo Carrillo",
    "Paulo Dino Gazzaniga"
]

minutes_test[
    minutes_test["player_name"].isin(players_to_check)
]

,team_name,player_name,minutes_played,position_changes
0,Elche,Guido Marcelo Carrillo,21.600000,0
1,Elche,Paulo Dino Gazzaniga,0.000000,0
10,Elche,Fidel Chaves de la Torre,33.650000,1
25,Barcelona,Jordi Alba Ramos,76.133333,0
30,Barcelona,Lionel Andrés Messi Cuccittini,92.983333,1
33,Barcelona,Miralem Pjanić,45.000000,0


In [38]:
position_interval_checks = []

for team in lineup_data:

    for player in team["lineup"]:

        positions = player.get("positions", [])

        if len(positions) > 1:

            for i in range(len(positions) - 1):

                current_to = positions[i].get("to")
                next_from = positions[i + 1].get("from")

                position_interval_checks.append({
                    "team_name": team["team_name"],
                    "player_name": player["player_name"],
                    "current_position": positions[i].get("position"),
                    "next_position": positions[i + 1].get("position"),
                    "current_to": current_to,
                    "next_from": next_from,
                    "continuous": current_to == next_from
                })

interval_check_df = pd.DataFrame(
    position_interval_checks
)

interval_check_df

,team_name,player_name,current_position,next_position,current_to,next_from,continuous
0,Elche,Johan Andrés Mojica Palacio,Left Wing Back,Left Wing Back,37:02,37:35,False
1,Elche,Fidel Chaves de la Torre,Right Wing Back,Left Wing,65:03,65:03,True
2,Elche,José Antonio Morente Oliva,Left Wing,Right Wing Back,65:03,65:03,True
3,Barcelona,Martin Braithwaite Christensen,Left Wing,Center Forward,45:00,45:00,True
4,Barcelona,Ousmane Dembélé,Left Wing,Center Defensive Midfield,45:00,45:00,True
5,Barcelona,Samuel Yves Umtiti,Left Center Back,Left Center Back,84:54,85:27,False
6,Barcelona,Lionel Andrés Messi Cuccittini,Center Forward,Center Attacking Midfield,45:00,45:00,True
7,Barcelona,Frenkie de Jong,Right Center Midfield,Left Center Midfield,45:00,45:00,True
8,Barcelona,Pedro González López,Left Center Midfield,Right Center Midfield,45:00,45:00,True


In [39]:
print(
    interval_check_df["continuous"].value_counts(
        dropna=False
    )
)

interval_check_df[
    interval_check_df["continuous"] == False
]

continuous
True     7
False    2
Name: count, dtype: int64


,team_name,player_name,current_position,next_position,current_to,next_from,continuous
0,Elche,Johan Andrés Mojica Palacio,Left Wing Back,Left Wing Back,37:02,37:35,False
5,Barcelona,Samuel Yves Umtiti,Left Center Back,Left Center Back,84:54,85:27,False


In [40]:
players_to_inspect = [
    "Johan Andrés Mojica Palacio",
    "Samuel Yves Umtiti"
]

for team in lineup_data:
    for player in team["lineup"]:

        if player["player_name"] in players_to_inspect:

            print("\nPLAYER:", player["player_name"])

            for position in player["positions"]:
                print(position)


PLAYER: Johan Andrés Mojica Palacio
{'position_id': 8, 'position': 'Left Wing Back', 'from': '00:00', 'to': '37:02', 'from_period': 1, 'to_period': 1, 'start_reason': 'Starting XI', 'end_reason': 'Player Off'}
{'position_id': 8, 'position': 'Left Wing Back', 'from': '37:35', 'to': None, 'from_period': 1, 'to_period': None, 'start_reason': 'Player On', 'end_reason': 'Final Whistle'}

PLAYER: Samuel Yves Umtiti
{'position_id': 5, 'position': 'Left Center Back', 'from': '00:00', 'to': '84:54', 'from_period': 1, 'to_period': 2, 'start_reason': 'Starting XI', 'end_reason': 'Player Off'}
{'position_id': 5, 'position': 'Left Center Back', 'from': '85:27', 'to': None, 'from_period': 2, 'to_period': None, 'start_reason': 'Player On', 'end_reason': 'Final Whistle'}


In [41]:
def calculate_player_minutes(lineup_data, match_end_seconds):

    player_minutes = []

    for team in lineup_data:

        team_name = team["team_name"]

        for player in team["lineup"]:

            positions = player.get("positions", [])

            if len(positions) == 0:

                player_minutes.append({
                    "team_name": team_name,
                    "player_name": player["player_name"],
                    "minutes_played": 0.0,
                    "position_changes": 0,
                    "temporary_off_pitch_gaps": 0
                })

                continue

            total_seconds = 0
            position_changes = 0
            temporary_off_pitch_gaps = 0

            for i, position in enumerate(positions):

                start_time = position.get("from")
                end_time = position.get("to")

                start_seconds = clock_to_seconds(
                    start_time
                )

                if pd.isna(end_time):
                    end_seconds = match_end_seconds
                else:
                    end_seconds = clock_to_seconds(
                        end_time
                    )

                if (
                    pd.notna(start_seconds)
                    and pd.notna(end_seconds)
                    and end_seconds >= start_seconds
                ):
                    total_seconds += (
                        end_seconds - start_seconds
                    )

                # Compare this interval with the next one
                if i < len(positions) - 1:

                    current_position = position.get(
                        "position"
                    )

                    next_position = positions[
                        i + 1
                    ].get("position")

                    if (
                        current_position
                        != next_position
                    ):
                        position_changes += 1

                    current_end = position.get("to")

                    next_start = positions[
                        i + 1
                    ].get("from")

                    if (
                        pd.notna(current_end)
                        and pd.notna(next_start)
                        and clock_to_seconds(next_start)
                        > clock_to_seconds(current_end)
                    ):
                        temporary_off_pitch_gaps += 1

            minutes_played = (
                total_seconds / 60
            )

            player_minutes.append({
                "team_name": team_name,
                "player_name": player["player_name"],
                "minutes_played": minutes_played,
                "position_changes": position_changes,
                "temporary_off_pitch_gaps":
                    temporary_off_pitch_gaps
            })

    return pd.DataFrame(player_minutes)

In [42]:
minutes_test = calculate_player_minutes(
    lineup_data,
    match_end_seconds
)

minutes_test[
    minutes_test["player_name"].isin([
        "Lionel Andrés Messi Cuccittini",
        "Fidel Chaves de la Torre",
        "Johan Andrés Mojica Palacio",
        "Samuel Yves Umtiti",
        "Jordi Alba Ramos",
        "Miralem Pjanić",
        "Guido Marcelo Carrillo"
    ])
]

,team_name,player_name,minutes_played,position_changes,temporary_off_pitch_gaps
0,Elche,Guido Marcelo Carrillo,21.600000,0,0
2,Elche,Johan Andrés Mojica Palacio,92.433333,0,1
10,Elche,Fidel Chaves de la Torre,33.650000,1,0
25,Barcelona,Jordi Alba Ramos,76.133333,0,0
29,Barcelona,Samuel Yves Umtiti,92.433333,0,1
30,Barcelona,Lionel Andrés Messi Cuccittini,92.983333,1,0
33,Barcelona,Miralem Pjanić,45.000000,0,0


In [43]:
print(
    "Players in lineup:",
    len(minutes_test)
)

print(
    "Negative minutes:",
    (minutes_test["minutes_played"] < 0).sum()
)

print(
    "Minutes greater than match duration:",
    (
        minutes_test["minutes_played"]
        > match_end_minutes
    ).sum()
)

print(
    "Missing minutes:",
    minutes_test["minutes_played"].isna().sum()
)

print(
    "Players with position changes:",
    (
        minutes_test["position_changes"] > 0
    ).sum()
)

print(
    "Players with temporary off-pitch gaps:",
    (
        minutes_test["temporary_off_pitch_gaps"] > 0
    ).sum()
)

print(
    "Unused players:",
    (
        minutes_test["minutes_played"] == 0
    ).sum()
)

Players in lineup: 45
Negative minutes: 0
Minutes greater than match duration: 0
Missing minutes: 0
Players with position changes: 7
Players with temporary off-pitch gaps: 2
Unused players: 13


In [44]:
def prepare_match_minutes(match_id):

    events_file = Path(
        f"open-data-master/data/events/{match_id}.json"
    )

    lineup_file = Path(
        f"open-data-master/data/lineups/{match_id}.json"
    )

    # -----------------------------
    # Load Events
    # -----------------------------

    with open(events_file, "r", encoding="utf-8") as f:
        events = json.load(f)

    events_df = pd.DataFrame(events)

    # -----------------------------
    # Calculate actual match end
    # -----------------------------

    max_minute = events_df["minute"].max()

    max_second = events_df.loc[
        events_df["minute"] == max_minute,
        "second"
    ].max()

    match_end_seconds = (
        max_minute * 60
        + max_second
    )

    # -----------------------------
    # Load Lineup
    # -----------------------------

    with open(lineup_file, "r", encoding="utf-8") as f:
        lineup_data = json.load(f)

    # -----------------------------
    # Calculate player minutes
    # -----------------------------

    minutes_df = calculate_player_minutes(
        lineup_data,
        match_end_seconds
    )

    minutes_df["match_id"] = int(match_id)

    minutes_df["match_duration_minutes"] = (
        match_end_seconds / 60
    )

    return minutes_df


In [45]:
minutes_wrapper_test = prepare_match_minutes(
    3764440
)

print(minutes_wrapper_test.shape)

minutes_wrapper_test[
    minutes_wrapper_test["player_name"].isin([
        "Lionel Andrés Messi Cuccittini",
        "Fidel Chaves de la Torre",
        "Johan Andrés Mojica Palacio",
        "Samuel Yves Umtiti"
    ])
]

(45, 7)


,team_name,player_name,minutes_played,position_changes,temporary_off_pitch_gaps,match_id,match_duration_minutes
2,Elche,Johan Andrés Mojica Palacio,92.433333,0,1,3764440,92.983333
10,Elche,Fidel Chaves de la Torre,33.650000,1,0,3764440,92.983333
29,Barcelona,Samuel Yves Umtiti,92.433333,0,1,3764440,92.983333
30,Barcelona,Lionel Andrés Messi Cuccittini,92.983333,1,0,3764440,92.983333


In [46]:
all_match_minutes = []
minutes_failures = []

for i, lineup_file in enumerate(
    lineup_files,
    start=1
):

    match_id = lineup_file.stem

    try:

        match_minutes = prepare_match_minutes(
            match_id
        )

        all_match_minutes.append(
            match_minutes
        )

    except Exception as e:

        minutes_failures.append({
            "match_id": match_id,
            "error": str(e)
        })

    if i % 500 == 0:
        print(
            f"Processed {i} / {len(lineup_files)} matches"
        )

print("\nFinished")
print(
    "Successful matches:",
    len(all_match_minutes)
)
print(
    "Failed matches:",
    len(minutes_failures)
)

Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches

Finished
Successful matches: 4235
Failed matches: 0


In [47]:
all_match_minutes = []
minutes_failures = []

for i, lineup_file in enumerate(
    lineup_files,
    start=1
):

    match_id = lineup_file.stem

    try:
        match_minutes = prepare_match_minutes(
            match_id
        )

        all_match_minutes.append(
            match_minutes
        )

    except Exception as e:
        minutes_failures.append({
            "match_id": match_id,
            "error": str(e)
        })

    if i % 500 == 0:
        print(
            f"Processed {i} / {len(lineup_files)} matches"
        )

print("\nFinished")

print(
    "Successful matches:",
    len(all_match_minutes)
)

print(
    "Failed matches:",
    len(minutes_failures)
)

Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches

Finished
Successful matches: 4235
Failed matches: 0


In [48]:
all_player_minutes = pd.concat(
    all_match_minutes,
    ignore_index=True
)

print(
    "Shape:",
    all_player_minutes.shape
)

print(
    "Unique matches:",
    all_player_minutes["match_id"].nunique()
)

print(
    "Unique players:",
    all_player_minutes["player_name"].nunique()
)

all_player_minutes.head()

Shape: (161958, 7)
Unique matches: 4235
Unique players: 11924


,team_name,player_name,minutes_played,position_changes,temporary_off_pitch_gaps,match_id,match_duration_minutes
0,Barcelona,Malcom Filipe Silva de Oliveira,0.000000,0,0,15946,92.516667
1,Barcelona,Philippe Coutinho Correia,47.516667,2,0,15946,92.516667
2,Barcelona,Sergio Busquets i Burgos,84.216667,1,0,15946,92.516667
3,Barcelona,Jordi Alba Ramos,92.516667,0,0,15946,92.516667
4,Barcelona,Gerard Piqué Bernabéu,92.516667,0,0,15946,92.516667


In [49]:
print(
    "Duplicate player-match rows:",
    all_player_minutes.duplicated(
        subset=["match_id", "player_name"]
    ).sum()
)

print(
    "Missing minutes:",
    all_player_minutes[
        "minutes_played"
    ].isna().sum()
)

print(
    "Negative minutes:",
    (
        all_player_minutes["minutes_played"] < 0
    ).sum()
)

print(
    "Minutes greater than match duration:",
    (
        all_player_minutes["minutes_played"]
        >
        all_player_minutes["match_duration_minutes"]
    ).sum()
)

print(
    "Zero-minute players:",
    (
        all_player_minutes["minutes_played"] == 0
    ).sum()
)

print(
    "Players with position changes:",
    (
        all_player_minutes["position_changes"] > 0
    ).sum()
)

print(
    "Players with temporary off-pitch gaps:",
    (
        all_player_minutes[
            "temporary_off_pitch_gaps"
        ] > 0
    ).sum()
)

Duplicate player-match rows: 2
Missing minutes: 0
Negative minutes: 0
Minutes greater than match duration: 176
Zero-minute players: 40745
Players with position changes: 32364
Players with temporary off-pitch gaps: 4408


In [50]:
all_player_minutes[
    [
        "match_id",
        "match_duration_minutes"
    ]
].drop_duplicates()["match_duration_minutes"].describe()

count    4235.000000
mean       94.799193
std         4.265056
min        51.683333
25%        93.083333
50%        94.066667
75%        95.266667
max       139.150000
Name: match_duration_minutes, dtype: float64

In [51]:
all_player_minutes[
    [
        "match_id",
        "match_duration_minutes"
    ]
].drop_duplicates().sort_values(
    "match_duration_minutes"
).head(10)

,match_id,match_duration_minutes
84950,3881507,51.683333
89879,3888787,85.183333
89915,3888854,86.866667
16268,3750180,87.066667
89356,3888701,88.100000
89580,3888713,88.566667
148454,68330,88.633333
89604,3888716,88.683333
156952,7477,88.750000
83581,3879867,89.300000


In [52]:
all_player_minutes[
    [
        "match_id",
        "match_duration_minutes"
    ]
].drop_duplicates().sort_values(
    "match_duration_minutes",
    ascending=False
).head(10)

,match_id,match_duration_minutes
114856,3902968,139.150000
140100,3922240,134.416667
140603,3923880,132.366667
147722,4018357,131.333333
147630,4018355,130.200000
114584,3901797,129.983333
140192,3922242,129.833333
145891,3942229,129.600000
57118,3827767,129.583333
114449,3901735,128.900000


In [53]:
check_match_id = 3902968

check_events_file = Path(
    f"open-data-master/data/events/{check_match_id}.json"
)

with open(
    check_events_file,
    "r",
    encoding="utf-8"
) as f:
    check_events = json.load(f)

check_events_df = pd.DataFrame(
    check_events
)

print(
    "Periods:",
    sorted(
        check_events_df["period"]
        .dropna()
        .unique()
    )
)

print(
    "\nMaximum minute by period:"
)

print(
    check_events_df
    .groupby("period")["minute"]
    .max()
)

Periods: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Maximum minute by period:
period
1     46
2     94
3    106
4    125
5    139
Name: minute, dtype: int64


In [54]:
check_events_df[
    [
        "period",
        "timestamp",
        "minute",
        "second",
        "type"
    ]
].tail(20)

,period,timestamp,minute,second,type
3856,5,00:10:47.035,130,47,"{'id': 16, 'name': 'Shot'}"
3857,5,00:10:47.670,130,47,"{'id': 23, 'name': 'Goal Keeper'}"
3858,5,00:11:30.518,131,30,"{'id': 16, 'name': 'Shot'}"
3859,5,00:11:31.067,131,31,"{'id': 23, 'name': 'Goal Keeper'}"
3860,5,00:12:25.363,132,25,"{'id': 16, 'name': 'Shot'}"
3861,5,00:12:25.942,132,25,"{'id': 23, 'name': 'Goal Keeper'}"
3862,5,00:13:17.072,133,17,"{'id': 16, 'name': 'Shot'}"
3863,5,00:13:17.511,133,17,"{'id': 23, 'name': 'Goal Keeper'}"
3864,5,00:14:12.480,134,12,"{'id': 16, 'name': 'Shot'}"
3865,5,00:14:13.048,134,13,"{'id': 23, 'name': 'Goal Keeper'}"


In [55]:
shootout_test = prepare_match_minutes(
    3902968
)

print(
    "Calculated match duration:",
    shootout_test[
        "match_duration_minutes"
    ].iloc[0]
)

Calculated match duration: 139.15


In [56]:
print(
    check_events_df
    .groupby("period")
    .agg(
        min_minute=("minute", "min"),
        max_minute=("minute", "max"),
        min_second=("second", "min"),
        max_second=("second", "max")
    )
)

        min_minute  max_minute  min_second  max_second
period                                                
1                0          46           0          59
2               45          94           0          59
3               90         106           0          59
4              105         125           0          59
5              120         139           0          55


In [57]:

playing_events_test = check_events_df[
    check_events_df["period"].isin(
        [1, 2, 3, 4]
    )
].copy()

print(
    "Periods retained:",
    sorted(
        playing_events_test[
            "period"
        ].unique()
    )
)

print(
    "Maximum retained minute:",
    playing_events_test[
        "minute"
    ].max()
)

playing_events_test[
    [
        "period",
        "timestamp",
        "minute",
        "second",
        "type"
    ]
].tail(10)

Periods retained: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Maximum retained minute: 125


,period,timestamp,minute,second,type
3821,4,00:19:36.285,124,36,"{'id': 23, 'name': 'Goal Keeper'}"
3822,4,00:19:38.412,124,38,"{'id': 2, 'name': 'Ball Recovery'}"
3823,4,00:19:38.412,124,38,"{'id': 43, 'name': 'Carry'}"
3824,4,00:19:39.828,124,39,"{'id': 30, 'name': 'Pass'}"
3825,4,00:19:40.060,124,40,"{'id': 42, 'name': 'Ball Receipt*'}"
3826,4,00:19:40.060,124,40,"{'id': 6, 'name': 'Block'}"
3827,4,00:20:17.139,125,17,"{'id': 30, 'name': 'Pass'}"
3828,4,00:20:19.188,125,19,"{'id': 9, 'name': 'Clearance'}"
3829,4,00:20:21.584,125,21,"{'id': 34, 'name': 'Half End'}"
3830,4,00:20:21.584,125,21,"{'id': 34, 'name': 'Half End'}"


In [58]:
def prepare_match_minutes(match_id):

    events_file = Path(
        f"open-data-master/data/events/{match_id}.json"
    )

    lineup_file = Path(
        f"open-data-master/data/lineups/{match_id}.json"
    )

    with open(
        events_file,
        "r",
        encoding="utf-8"
    ) as f:
        events = json.load(f)

    events_df = pd.DataFrame(events)

    # Keep only normal playing periods:
    # 1-2 = regulation
    # 3-4 = extra time
    # 5 = penalty shootout, excluded from minutes played
    playing_events = events_df[
        events_df["period"].isin([1, 2, 3, 4])
    ].copy()

    max_minute = playing_events["minute"].max()

    max_second = playing_events.loc[
        playing_events["minute"] == max_minute,
        "second"
    ].max()

    match_end_seconds = (
        max_minute * 60
        + max_second
    )

    with open(
        lineup_file,
        "r",
        encoding="utf-8"
    ) as f:
        lineup_data = json.load(f)

    minutes_df = calculate_player_minutes(
        lineup_data,
        match_end_seconds
    )

    minutes_df["match_id"] = int(match_id)

    minutes_df["match_duration_minutes"] = (
        match_end_seconds / 60
    )

    return minutes_df

In [59]:
shootout_test = prepare_match_minutes(
    3902968
)

print(
    "Calculated match duration:",
    shootout_test[
        "match_duration_minutes"
    ].iloc[0]
)

Calculated match duration: 125.35


In [60]:
all_match_minutes = []
minutes_failures = []

for i, lineup_file in enumerate(
    lineup_files,
    start=1
):

    match_id = lineup_file.stem

    try:
        match_minutes = prepare_match_minutes(
            match_id
        )

        all_match_minutes.append(
            match_minutes
        )

    except Exception as e:
        minutes_failures.append({
            "match_id": match_id,
            "error": str(e)
        })

    if i % 500 == 0:
        print(
            f"Processed {i} / {len(lineup_files)} matches"
        )

print("\nFinished")
print(
    "Successful matches:",
    len(all_match_minutes)
)
print(
    "Failed matches:",
    len(minutes_failures)
)

Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches

Finished
Successful matches: 4235
Failed matches: 0


In [62]:
all_player_minutes = pd.concat(
    all_match_minutes,
    ignore_index=True
)

print("Shape:", all_player_minutes.shape)
print(
    "Unique matches:",
    all_player_minutes["match_id"].nunique()
)

Shape: (161958, 7)
Unique matches: 4235


In [63]:
print(
    "Duplicate player-match rows:",
    all_player_minutes.duplicated(
        subset=["match_id", "player_name"]
    ).sum()
)

print(
    "Missing minutes:",
    all_player_minutes[
        "minutes_played"
    ].isna().sum()
)

print(
    "Negative minutes:",
    (
        all_player_minutes[
            "minutes_played"
        ] < 0
    ).sum()
)

print(
    "Minutes greater than match duration:",
    (
        all_player_minutes["minutes_played"]
        >
        all_player_minutes["match_duration_minutes"]
    ).sum()
)

print(
    "Zero-minute players:",
    (
        all_player_minutes[
            "minutes_played"
        ] == 0
    ).sum()
)

print(
    "Players with position changes:",
    (
        all_player_minutes[
            "position_changes"
        ] > 0
    ).sum()
)

print(
    "Players with temporary off-pitch gaps:",
    (
        all_player_minutes[
            "temporary_off_pitch_gaps"
        ] > 0
    ).sum()
)

Duplicate player-match rows: 2
Missing minutes: 0
Negative minutes: 0
Minutes greater than match duration: 176
Zero-minute players: 40745
Players with position changes: 32364
Players with temporary off-pitch gaps: 4408


In [64]:
match_durations = (
    all_player_minutes[
        ["match_id", "match_duration_minutes"]
    ]
    .drop_duplicates()
)

match_durations[
    "match_duration_minutes"
].describe()

count    4235.000000
mean       94.714959
std         3.749593
min        51.683333
25%        93.083333
50%        94.066667
75%        95.266667
max       127.633333
Name: match_duration_minutes, dtype: float64

In [65]:
match_durations.sort_values(
    "match_duration_minutes",
    ascending=False
).head(15)

,match_id,match_duration_minutes
153597,69284,127.633333
145736,3942226,126.100000
114856,3902968,125.350000
146438,3943077,124.466667
38693,3794692,124.433333
160526,8656,124.383333
140466,3922659,124.333333
66966,3869685,124.116667
63462,3844384,124.066667
140100,3922240,123.933333


In [66]:
check_match_id = 69284

check_file = Path(
    f"open-data-master/data/events/{check_match_id}.json"
)

with open(
    check_file,
    "r",
    encoding="utf-8"
) as f:
    check_events = json.load(f)

check_df = pd.DataFrame(check_events)

print(
    check_df
    .groupby("period")
    .agg(
        min_minute=("minute", "min"),
        max_minute=("minute", "max")
    )
)

        min_minute  max_minute
period                        
1                0          48
2               45          95
3               90         106
4              105         127


In [67]:
check_df[
    [
        "period",
        "timestamp",
        "minute",
        "second",
        "type"
    ]
].tail(15)

,period,timestamp,minute,second,type
4227,4,00:22:15.945,127,15,"{'id': 42, 'name': 'Ball Receipt*'}"
4228,4,00:22:15.945,127,15,"{'id': 43, 'name': 'Carry'}"
4229,4,00:22:16.752,127,16,"{'id': 3, 'name': 'Dispossessed'}"
4230,4,00:22:16.752,127,16,"{'id': 4, 'name': 'Duel'}"
4231,4,00:22:26.778,127,26,"{'id': 30, 'name': 'Pass'}"
4232,4,00:22:27.964,127,27,"{'id': 17, 'name': 'Pressure'}"
4233,4,00:22:28.225,127,28,"{'id': 42, 'name': 'Ball Receipt*'}"
4234,4,00:22:28.225,127,28,"{'id': 43, 'name': 'Carry'}"
4235,4,00:22:29.280,127,29,"{'id': 30, 'name': 'Pass'}"
4236,4,00:22:30.391,127,30,"{'id': 42, 'name': 'Ball Receipt*'}"


In [68]:
print(
    "Duplicate player-match rows:",
    all_player_minutes.duplicated(
        subset=["match_id", "player_name"]
    ).sum()
)

print(
    "Missing minutes:",
    all_player_minutes["minutes_played"]
    .isna().sum()
)

print(
    "Negative minutes:",
    (
        all_player_minutes["minutes_played"] < 0
    ).sum()
)

print(
    "Minutes greater than match duration:",
    (
        all_player_minutes["minutes_played"]
        >
        all_player_minutes["match_duration_minutes"]
    ).sum()
)

print(
    "Zero-minute players:",
    (
        all_player_minutes["minutes_played"] == 0
    ).sum()
)

print(
    "Players with position changes:",
    (
        all_player_minutes["position_changes"] > 0
    ).sum()
)

print(
    "Players with temporary off-pitch gaps:",
    (
        all_player_minutes[
            "temporary_off_pitch_gaps"
        ] > 0
    ).sum()
)

Duplicate player-match rows: 2
Missing minutes: 0
Negative minutes: 0
Minutes greater than match duration: 176
Zero-minute players: 40745
Players with position changes: 32364
Players with temporary off-pitch gaps: 4408


In [69]:
duplicate_player_matches = (
    all_player_minutes[
        all_player_minutes.duplicated(
            subset=[
                "match_id",
                "player_name"
            ],
            keep=False
        )
    ]
    .sort_values(
        ["match_id", "player_name"]
    )
)

duplicate_player_matches

,team_name,player_name,minutes_played,position_changes,temporary_off_pitch_gaps,match_id,match_duration_minutes
41776,ATK Mohun Bagan,Manvir Singh,95.133333,0,0,3813306,95.133333
41804,NorthEast United,Manvir Singh,0.000000,0,0,3813306,95.133333
43297,NorthEast United,Manvir Singh,0.000000,0,0,3817873,94.033333
43305,Mohun Bagan Super Giant,Manvir Singh,94.033333,0,0,3817873,94.033333


In [70]:
minutes_over_duration = (
    all_player_minutes[
        all_player_minutes["minutes_played"]
        >
        all_player_minutes[
            "match_duration_minutes"
        ]
    ]
    .copy()
)

minutes_over_duration[
    "excess_minutes"
] = (
    minutes_over_duration["minutes_played"]
    -
    minutes_over_duration[
        "match_duration_minutes"
    ]
)

print(
    "Affected player records:",
    len(minutes_over_duration)
)

print(
    "Affected matches:",
    minutes_over_duration[
        "match_id"
    ].nunique()
)

minutes_over_duration[
    [
        "match_id",
        "team_name",
        "player_name",
        "minutes_played",
        "match_duration_minutes",
        "excess_minutes",
        "position_changes",
        "temporary_off_pitch_gaps"
    ]
].sort_values(
    "excess_minutes",
    ascending=False
).head(30)

Affected player records: 176
Affected matches: 50


,match_id,team_name,player_name,minutes_played,match_duration_minutes,excess_minutes,position_changes,temporary_off_pitch_gaps
147759,4018357,Germany Women's,Jule Brand,226.350000,123.033333,103.316667,0,0
147753,4018357,Germany Women's,Elisa Senß,225.850000,123.033333,102.816667,1,0
146442,3943077,Argentina,Nicolás Alejandro Tagliafico,224.950000,124.466667,100.483333,2,0
146447,3943077,Argentina,Rodrigo Javier De Paul,224.950000,124.466667,100.483333,3,0
146438,3943077,Argentina,Ángel Fabián Di María Hernández,224.583333,124.466667,100.116667,2,0
147725,4018357,France Women's,Onema Grace Geyoro,219.483333,123.033333,96.450000,1,0
147766,4018357,Germany Women's,Franziska Kett,219.150000,123.033333,96.116667,0,1
147732,4018357,France Women's,Elisa De Almeida,218.883333,123.033333,95.850000,0,0
63475,3844384,England Women's,Lauren Hemp,215.300000,124.066667,91.233333,4,0
136447,3913146,Bristol City WFC,Ffion Morgan,193.066667,102.016667,91.050000,3,0


In [71]:
for check_match_id in [3813306, 3817873]:

    lineup_file = Path(
        f"open-data-master/data/lineups/{check_match_id}.json"
    )

    with open(
        lineup_file,
        "r",
        encoding="utf-8"
    ) as f:
        lineup_check = json.load(f)

    print(
        "\nMATCH:",
        check_match_id
    )

    for team in lineup_check:

        for player in team["lineup"]:

            if player["player_name"] == "Manvir Singh":

                print(
                    team["team_name"],
                    "|",
                    player["player_name"],
                    "| player_id:",
                    player["player_id"]
                )


MATCH: 3813306
ATK Mohun Bagan | Manvir Singh | player_id: 124741
NorthEast United | Manvir Singh | player_id: 162757

MATCH: 3817873
NorthEast United | Manvir Singh | player_id: 162757
Mohun Bagan Super Giant | Manvir Singh | player_id: 124741


In [72]:
check_match_id = 4018357
check_player = "Jule Brand"

lineup_file = Path(
    f"open-data-master/data/lineups/{check_match_id}.json"
)

with open(
    lineup_file,
    "r",
    encoding="utf-8"
) as f:
    lineup_check = json.load(f)

for team in lineup_check:

    for player in team["lineup"]:

        if player["player_name"] == check_player:

            print("Team:", team["team_name"])
            print("Player:", player["player_name"])
            print("Player ID:", player["player_id"])
            print()

            for i, position in enumerate(
                player.get("positions", [])
            ):

                print(f"Interval {i + 1}")
                print("Position:", position.get("position"))
                print("From:", position.get("from"))
                print("To:", position.get("to"))
                print("From period:", position.get("from_period"))
                print("To period:", position.get("to_period"))
                print("Start reason:", position.get("start_reason"))
                print("End reason:", position.get("end_reason"))
                print("-" * 40)

Team: Germany Women's
Player: Jule Brand
Player ID: 46987

Interval 1
Position: Right Midfield
From: 00:00
To: 119:41
From period: 1
To period: 4
Start reason: Starting XI
End reason: Substitution - Off (Tactical)
----------------------------------------
Interval 2
Position: Right Midfield
From: 16:22
To: None
From period: 1
To period: None
Start reason: Tactical Shift
End reason: Final Whistle
----------------------------------------


In [74]:
def calculate_player_minutes(
    lineup_data,
    match_end_seconds
):

    player_minutes = []

    for team in lineup_data:

        team_name = team["team_name"]

        for player in team["lineup"]:

            positions = player.get(
                "positions",
                []
            )

            player_id = player.get(
                "player_id"
            )

            if len(positions) == 0:

                player_minutes.append({
                    "team_name": team_name,
                    "player_id": player_id,
                    "player_name": player["player_name"],
                    "minutes_played": 0.0,
                    "position_changes": 0,
                    "temporary_off_pitch_gaps": 0
                })

                continue

            intervals = []
            position_changes = 0
            temporary_off_pitch_gaps = 0

            for i, position in enumerate(
                positions
            ):

                start_time = position.get("from")
                end_time = position.get("to")

                start_seconds = clock_to_seconds(
                    start_time
                )

                if pd.isna(end_time):
                    end_seconds = match_end_seconds
                else:
                    end_seconds = clock_to_seconds(
                        end_time
                    )

                if (
                    pd.notna(start_seconds)
                    and pd.notna(end_seconds)
                ):

                    start_seconds = max(
                        0,
                        start_seconds
                    )

                    end_seconds = min(
                        match_end_seconds,
                        end_seconds
                    )

                    if end_seconds >= start_seconds:

                        intervals.append(
                            (
                                start_seconds,
                                end_seconds
                            )
                        )

                if i < len(positions) - 1:

                    current_position = (
                        position.get("position")
                    )

                    next_position = (
                        positions[i + 1].get(
                            "position"
                        )
                    )

                    if current_position != next_position:
                        position_changes += 1

                    current_end = position.get("to")
                    next_start = positions[i + 1].get(
                        "from"
                    )

                    if (
                        pd.notna(current_end)
                        and pd.notna(next_start)
                    ):

                        current_end_seconds = (
                            clock_to_seconds(
                                current_end
                            )
                        )

                        next_start_seconds = (
                            clock_to_seconds(
                                next_start
                            )
                        )

                        if (
                            next_start_seconds
                            > current_end_seconds
                        ):
                            temporary_off_pitch_gaps += 1

            # Sort intervals by their starting time
            intervals.sort(
                key=lambda x: x[0]
            )

            # Merge overlapping intervals
            merged_intervals = []

            for start, end in intervals:

                if not merged_intervals:

                    merged_intervals.append(
                        [start, end]
                    )

                else:

                    last_start, last_end = (
                        merged_intervals[-1]
                    )

                    if start <= last_end:

                        merged_intervals[-1][1] = max(
                            last_end,
                            end
                        )

                    else:

                        merged_intervals.append(
                            [start, end]
                        )

            # Sum only the merged intervals
            total_seconds = sum(
                end - start
                for start, end in merged_intervals
            )

            minutes_played = (
                total_seconds / 60
            )

            player_minutes.append({
                "team_name": team_name,
                "player_id": player_id,
                "player_name": player["player_name"],
                "minutes_played": minutes_played,
                "position_changes": position_changes,
                "temporary_off_pitch_gaps":
                    temporary_off_pitch_gaps
            })

    return pd.DataFrame(
        player_minutes
    )

In [75]:
def prepare_match_minutes(match_id):

    events_file = Path(
        f"open-data-master/data/events/{match_id}.json"
    )

    lineup_file = Path(
        f"open-data-master/data/lineups/{match_id}.json"
    )

    with open(
        events_file,
        "r",
        encoding="utf-8"
    ) as f:
        events = json.load(f)

    events_df = pd.DataFrame(events)

    # Periods 1-2 = regulation
    # Periods 3-4 = extra time
    # Period 5 = penalty shootout, excluded
    playing_events = events_df[
        events_df["period"].isin(
            [1, 2, 3, 4]
        )
    ].copy()

    max_minute = (
        playing_events["minute"].max()
    )

    max_second = (
        playing_events.loc[
            playing_events["minute"]
            == max_minute,
            "second"
        ].max()
    )

    match_end_seconds = (
        max_minute * 60
        + max_second
    )

    with open(
        lineup_file,
        "r",
        encoding="utf-8"
    ) as f:
        lineup_data = json.load(f)

    minutes_df = calculate_player_minutes(
        lineup_data,
        match_end_seconds
    )

    minutes_df["match_id"] = int(
        match_id
    )

    minutes_df[
        "match_duration_minutes"
    ] = (
        match_end_seconds / 60
    )

    return minutes_df

In [76]:
jule_test = prepare_match_minutes(
    4018357
)

jule_test[
    jule_test["player_name"]
    == "Jule Brand"
]

,team_name,player_id,player_name,minutes_played,position_changes,temporary_off_pitch_gaps,match_id,match_duration_minutes
37,Germany Women's,46987,Jule Brand,123.033333,0,0,4018357,123.033333


In [77]:
test_ids = [
    4018357,  # Jule Brand case
    3943077,  # Argentina case
    3844384,  # England Women's case
    3869685,  # Messi case
    3795108   # Spain case
]

validation_minutes = []

for match_id in test_ids:
    validation_minutes.append(
        prepare_match_minutes(match_id)
    )

validation_minutes = pd.concat(
    validation_minutes,
    ignore_index=True
)

validation_minutes[
    "exceeds_match_duration"
] = (
    validation_minutes["minutes_played"]
    >
    validation_minutes["match_duration_minutes"]
)

print(
    "Players exceeding match duration:",
    validation_minutes[
        "exceeds_match_duration"
    ].sum()
)

validation_minutes[
    validation_minutes[
        "exceeds_match_duration"
    ]
][
    [
        "match_id",
        "player_id",
        "player_name",
        "minutes_played",
        "match_duration_minutes"
    ]
]

Players exceeding match duration: 0


,match_id,player_id,player_name,minutes_played,match_duration_minutes


In [78]:
all_match_minutes = []
minutes_failures = []

for i, lineup_file in enumerate(
    lineup_files,
    start=1
):

    match_id = lineup_file.stem

    try:
        match_minutes = prepare_match_minutes(
            match_id
        )

        all_match_minutes.append(
            match_minutes
        )

    except Exception as e:
        minutes_failures.append({
            "match_id": match_id,
            "error": str(e)
        })

    if i % 500 == 0:
        print(
            f"Processed {i} / {len(lineup_files)} matches"
        )

print("\nFinished")
print(
    "Successful matches:",
    len(all_match_minutes)
)
print(
    "Failed matches:",
    len(minutes_failures)
)

Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches

Finished
Successful matches: 4235
Failed matches: 0


In [79]:
all_player_minutes = pd.concat(
    all_match_minutes,
    ignore_index=True
)

print(
    "Duplicate player-match rows:",
    all_player_minutes.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

print(
    "Missing minutes:",
    all_player_minutes[
        "minutes_played"
    ].isna().sum()
)

print(
    "Negative minutes:",
    (
        all_player_minutes[
            "minutes_played"
        ] < 0
    ).sum()
)

print(
    "Minutes greater than match duration:",
    (
        all_player_minutes["minutes_played"]
        >
        all_player_minutes["match_duration_minutes"]
    ).sum()
)

print(
    "Total player records:",
    len(all_player_minutes)
)

print(
    "Unique matches:",
    all_player_minutes[
        "match_id"
    ].nunique()
)

Duplicate player-match rows: 0
Missing minutes: 0
Negative minutes: 0
Minutes greater than match duration: 0
Total player records: 161958
Unique matches: 4235


In [80]:
all_player_minutes.to_pickle(
    "all_player_minutes_prepared.pkl"
)

print("Saved successfully")

Saved successfully


In [81]:
print(historical_clean.columns.tolist())

['match_id', 'player_name', 'total_events', 'pass_attempts', 'passes_completed', 'forward_passes', 'progressive_passes', 'avg_pass_length', 'avg_pass_angle_degrees', 'crosses', 'through_balls', 'switches', 'shot_assists', 'goal_assists', 'shots', 'carries', 'miscontrols', 'pass_completion_rate', 'match_date', 'home_team_name', 'away_team_name', 'prev_pass_completion_rate', 'rolling_5_pass_completion', 'rolling_3_pass_completion', 'prev_pass_attempts', 'rolling_3_pass_attempts', 'rolling_5_pass_attempts', 'prev_progressive_passes', 'rolling_3_progressive_passes', 'rolling_5_progressive_passes', 'prev_carries', 'rolling_3_carries', 'rolling_5_carries', 'prev_shots', 'rolling_3_shots', 'rolling_5_shots', 'prev_miscontrols', 'rolling_3_miscontrols', 'rolling_5_miscontrols', 'prev_shot_assists', 'rolling_3_shot_assists', 'rolling_5_shot_assists', 'previous_matches_available']


In [82]:
def prepare_historical_event_match(match_id):

    events_file = Path(
        f"open-data-master/data/events/{match_id}.json"
    )

    with open(events_file, "r", encoding="utf-8") as f:
        events = json.load(f)

    df = pd.DataFrame(events)

    df["match_id"] = int(match_id)

    for column in ["type", "player", "pass", "location"]:
        if column not in df.columns:
            df[column] = None

    # -----------------------------
    # Event type
    # -----------------------------
    df["event_type"] = df["type"].apply(
        lambda x: x.get("name")
        if isinstance(x, dict)
        else x
    )

    # -----------------------------
    # Player identifiers
    # -----------------------------
    df["player_id"] = df["player"].apply(
        lambda x: x.get("id")
        if isinstance(x, dict)
        else np.nan
    )

    df["player_name"] = df["player"].apply(
        lambda x: x.get("name")
        if isinstance(x, dict)
        else None
    )

    # -----------------------------
    # Basic event indicators
    # -----------------------------
    df["is_pass"] = (
        df["event_type"] == "Pass"
    )

    df["is_shot"] = (
        df["event_type"] == "Shot"
    )

    df["is_carry"] = (
        df["event_type"] == "Carry"
    )

    df["is_miscontrol"] = (
        df["event_type"] == "Miscontrol"
    )

    # -----------------------------
    # Pass completion
    # -----------------------------
    df["pass_completed"] = df["pass"].apply(
        lambda x:
            True
            if isinstance(x, dict)
            and x.get("outcome") is None
            else False
            if isinstance(x, dict)
            else pd.NA
    )

    # -----------------------------
    # Pass characteristics
    # -----------------------------
    df["pass_length"] = df["pass"].apply(
        lambda x: x.get("length", np.nan)
        if isinstance(x, dict)
        else np.nan
    )

    df["pass_angle"] = df["pass"].apply(
        lambda x: x.get("angle", np.nan)
        if isinstance(x, dict)
        else np.nan
    )

    df["pass_angle_degrees"] = np.degrees(
        pd.to_numeric(
            df["pass_angle"],
            errors="coerce"
        )
    )

    df["shot_assist"] = df["pass"].apply(
        lambda x: x.get("shot_assist", False)
        if isinstance(x, dict)
        else False
    )

    df["goal_assist"] = df["pass"].apply(
        lambda x: x.get("goal_assist", False)
        if isinstance(x, dict)
        else False
    )

    df["is_cross"] = df["pass"].apply(
        lambda x: x.get("cross", False)
        if isinstance(x, dict)
        else False
    )

    df["is_through_ball"] = df["pass"].apply(
        lambda x: x.get("through_ball", False)
        if isinstance(x, dict)
        else False
    )

    df["is_switch"] = df["pass"].apply(
        lambda x: x.get("switch", False)
        if isinstance(x, dict)
        else False
    )

    # -----------------------------
    # Starting location
    # -----------------------------
    df["start_x"] = df["location"].apply(
        lambda x: x[0]
        if isinstance(x, list)
        and len(x) >= 2
        else np.nan
    )

    df["start_y"] = df["location"].apply(
        lambda x: x[1]
        if isinstance(x, list)
        and len(x) >= 2
        else np.nan
    )

    # -----------------------------
    # Pass ending location
    # -----------------------------
    def get_pass_end_coordinate(
        pass_data,
        index
    ):

        if not isinstance(pass_data, dict):
            return np.nan

        end_location = pass_data.get(
            "end_location"
        )

        if (
            isinstance(end_location, list)
            and len(end_location) >= 2
        ):
            return end_location[index]

        return np.nan

    df["end_x"] = df["pass"].apply(
        lambda x:
            get_pass_end_coordinate(x, 0)
    )

    df["end_y"] = df["pass"].apply(
        lambda x:
            get_pass_end_coordinate(x, 1)
    )

    # -----------------------------
    # Forward passes
    # -----------------------------
    df["forward_distance"] = (
        df["end_x"]
        - df["start_x"]
    )

    df["is_forward_pass"] = (
        df["is_pass"]
        & (df["forward_distance"] > 0)
    )

    # -----------------------------
    # Distance to opponent goal
    # StatsBomb pitch = 120 x 80
    # Goal centre = (120, 40)
    # -----------------------------
    df["distance_to_goal_start"] = np.sqrt(
        (120 - df["start_x"]) ** 2
        +
        (40 - df["start_y"]) ** 2
    )

    df["distance_to_goal_end"] = np.sqrt(
        (120 - df["end_x"]) ** 2
        +
        (40 - df["end_y"]) ** 2
    )

    df["goal_distance_reduction"] = (
        df["distance_to_goal_start"]
        -
        df["distance_to_goal_end"]
    )

    df["goal_distance_reduction_pct"] = np.where(
        df["distance_to_goal_start"] > 0,
        (
            df["goal_distance_reduction"]
            /
            df["distance_to_goal_start"]
        ) * 100,
        np.nan
    )

    # Working operational definition:
    # progressive pass reduces goal distance
    # by at least 25%
    df["is_progressive_pass"] = (
        df["is_pass"]
        &
        (
            df[
                "goal_distance_reduction_pct"
            ] >= 25
        )
    )

    return df

In [83]:
historical_test = (
    prepare_historical_event_match(
        3764440
    )
)

historical_test[
    [
        "match_id",
        "player_id",
        "player_name",
        "event_type"
    ]
].dropna(
    subset=["player_id"]
).head(10)

,match_id,player_id,player_name,event_type
4,3764440,12072.0,Pere Milla Peña,Pass
5,3764440,24517.0,José Raúl Gutiérrez Parejo,Ball Receipt*
6,3764440,24517.0,José Raúl Gutiérrez Parejo,Carry
7,3764440,24517.0,José Raúl Gutiérrez Parejo,Pass
8,3764440,24169.0,Gonzalo Cacicedo Verdú,Ball Receipt*
9,3764440,24169.0,Gonzalo Cacicedo Verdú,Carry
10,3764440,24169.0,Gonzalo Cacicedo Verdú,Pass
11,3764440,9857.0,José Manuel Sánchez Guillén,Ball Receipt*
12,3764440,9857.0,José Manuel Sánchez Guillén,Carry
13,3764440,9857.0,José Manuel Sánchez Guillén,Pass


In [84]:
def aggregate_player_match(match_df):

    player_match = (
        match_df
        .dropna(
            subset=[
                "player_id",
                "player_name"
            ]
        )
        .groupby(
            [
                "match_id",
                "player_id",
                "player_name"
            ]
        )
        .agg(
            total_events=(
                "event_type",
                "count"
            ),
            pass_attempts=(
                "is_pass",
                "sum"
            ),
            passes_completed=(
                "pass_completed",
                "sum"
            ),
            forward_passes=(
                "is_forward_pass",
                "sum"
            ),
            progressive_passes=(
                "is_progressive_pass",
                "sum"
            ),
            avg_pass_length=(
                "pass_length",
                "mean"
            ),
            avg_pass_angle_degrees=(
                "pass_angle_degrees",
                "mean"
            ),
            crosses=(
                "is_cross",
                "sum"
            ),
            through_balls=(
                "is_through_ball",
                "sum"
            ),
            switches=(
                "is_switch",
                "sum"
            ),
            shot_assists=(
                "shot_assist",
                "sum"
            ),
            goal_assists=(
                "goal_assist",
                "sum"
            ),
            shots=(
                "is_shot",
                "sum"
            ),
            carries=(
                "is_carry",
                "sum"
            ),
            miscontrols=(
                "is_miscontrol",
                "sum"
            )
        )
        .reset_index()
    )

    player_match["pass_attempts"] = (
        pd.to_numeric(
            player_match[
                "pass_attempts"
            ],
            errors="coerce"
        )
    )

    player_match["passes_completed"] = (
        pd.to_numeric(
            player_match[
                "passes_completed"
            ],
            errors="coerce"
        )
    )

    player_match[
        "pass_completion_rate"
    ] = np.nan

    has_pass_attempts = (
        player_match[
            "pass_attempts"
        ] > 0
    )

    player_match.loc[
        has_pass_attempts,
        "pass_completion_rate"
    ] = (
        player_match.loc[
            has_pass_attempts,
            "passes_completed"
        ]
        /
        player_match.loc[
            has_pass_attempts,
            "pass_attempts"
        ]
    ) * 100

    return player_match

In [85]:
player_match_test = (
    aggregate_player_match(
        historical_test
    )
)

player_match_test[
    [
        "match_id",
        "player_id",
        "player_name",
        "pass_attempts",
        "passes_completed",
        "pass_completion_rate"
    ]
].head(10)

,match_id,player_id,player_name,pass_attempts,passes_completed,pass_completion_rate
0,3764440,3246.0,Guido Marcelo Carrillo,3,2,66.666667
1,3764440,4447.0,Martin Braithwaite Christensen,19,13,68.421053
2,3764440,5203.0,Sergio Busquets i Burgos,22,22,100.000000
3,3764440,5211.0,Jordi Alba Ramos,70,66,94.285714
4,3764440,5213.0,Gerard Piqué Bernabéu,92,85,92.391304
5,3764440,5477.0,Ousmane Dembélé,26,23,88.461538
6,3764440,5487.0,Antoine Griezmann,8,7,87.500000
7,3764440,5492.0,Samuel Yves Umtiti,95,90,94.736842
8,3764440,5503.0,Lionel Andrés Messi Cuccittini,72,60,83.333333
9,3764440,5691.0,Johan Andrés Mojica Palacio,46,41,89.130435


In [86]:
duplicate_ids = (
    player_match_test
    .duplicated(
        subset=[
            "match_id",
            "player_id"
        ]
    )
    .sum()
)

missing_player_ids = (
    player_match_test[
        "player_id"
    ]
    .isna()
    .sum()
)

print(
    "Player-match rows:",
    len(player_match_test)
)

print(
    "Duplicate match_id + player_id:",
    duplicate_ids
)

print(
    "Missing player IDs:",
    missing_player_ids
)

print(
    "Unique players:",
    player_match_test[
        "player_id"
    ].nunique()
)

Player-match rows: 32
Duplicate match_id + player_id: 0
Missing player IDs: 0
Unique players: 32


In [88]:
from pathlib import Path
import pandas as pd

events_folder = Path(
    "open-data-master/data/events"
)

event_files = list(
    events_folder.glob("*.json")
)

historical_matches = []
failed_historical_matches = []

total_matches = len(event_files)

print(
    "Events files found:",
    total_matches
)

for i, event_file in enumerate(
    event_files,
    start=1
):

    match_id = event_file.stem

    try:

        # Prepare event-level features
        match_df = (
            prepare_historical_event_match(
                match_id
            )
        )

        # Aggregate to player-match level
        player_match_df = (
            aggregate_player_match(
                match_df
            )
        )

        historical_matches.append(
            player_match_df
        )

    except Exception as e:

        failed_historical_matches.append({
            "match_id": match_id,
            "error": str(e)
        })

    if i % 500 == 0:
        print(
            f"Processed {i} / "
            f"{total_matches} matches"
        )

print("\nFinished")

print(
    "Successful matches:",
    len(historical_matches)
)

print(
    "Failed matches:",
    len(failed_historical_matches)
)

Events files found: 4235
Processed 500 / 4235 matches
Processed 1000 / 4235 matches
Processed 1500 / 4235 matches
Processed 2000 / 4235 matches
Processed 2500 / 4235 matches
Processed 3000 / 4235 matches
Processed 3500 / 4235 matches
Processed 4000 / 4235 matches

Finished
Successful matches: 4235
Failed matches: 0


In [90]:
historical_all = pd.concat(
    historical_matches,
    ignore_index=True
)

print(
    "Historical dataset created."
)

print(
    "Shape:",
    historical_all.shape
)

Historical dataset created.
Shape: (121034, 19)


In [91]:
print(
    "Unique matches:",
    historical_all["match_id"].nunique()
)

print(
    "Unique player IDs:",
    historical_all["player_id"].nunique()
)

print(
    "Duplicate match_id + player_id:",
    historical_all.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

print(
    "Missing player IDs:",
    historical_all["player_id"].isna().sum()
)

Unique matches: 4235
Unique player IDs: 9983
Duplicate match_id + player_id: 0
Missing player IDs: 0


In [93]:
from pathlib import Path
import json
import pandas as pd

matches_folder = Path(
    "open-data-master/data/matches"
)

metadata_records = []

match_files = list(
    matches_folder.rglob("*.json")
)

print(
    "Metadata files found:",
    len(match_files)
)

for match_file in match_files:

    try:

        with open(
            match_file,
            "r",
            encoding="utf-8"
        ) as f:
            matches = json.load(f)

        for match in matches:

            metadata_records.append({
                "match_id": match.get(
                    "match_id"
                ),
                "match_date": match.get(
                    "match_date"
                ),
                "home_team_name": (
                    match.get(
                        "home_team",
                        {}
                    ).get("home_team_name")
                ),
                "away_team_name": (
                    match.get(
                        "away_team",
                        {}
                    ).get("away_team_name")
                )
            })

    except Exception as e:

        print(
            "Problem reading:",
            match_file,
            e
        )

match_metadata = pd.DataFrame(
    metadata_records
)

# Standardise match ID
match_metadata["match_id"] = (
    pd.to_numeric(
        match_metadata["match_id"],
        errors="coerce"
    )
)

# Convert date to proper datetime
match_metadata["match_date"] = (
    pd.to_datetime(
        match_metadata["match_date"],
        errors="coerce"
    )
)

# Remove accidental duplicate metadata rows
match_metadata = (
    match_metadata
    .drop_duplicates(
        subset=["match_id"]
    )
    .reset_index(drop=True)
)

print(
    "Metadata shape:",
    match_metadata.shape
)

print(
    "Unique metadata matches:",
    match_metadata[
        "match_id"
    ].nunique()
)

print(
    "Missing dates:",
    match_metadata[
        "match_date"
    ].isna().sum()
)

print(
    match_metadata.columns.tolist()
)

Metadata files found: 80
Metadata shape: (3961, 4)
Unique metadata matches: 3961
Missing dates: 0
['match_id', 'match_date', 'home_team_name', 'away_team_name']


In [94]:
historical_with_metadata = historical_all.merge(
    match_metadata,
    on="match_id",
    how="left",
    validate="many_to_one"
)

print(
    "Merged shape:",
    historical_with_metadata.shape
)

print(
    "Unique matches:",
    historical_with_metadata[
        "match_id"
    ].nunique()
)

print(
    "Player-match rows missing dates:",
    historical_with_metadata[
        "match_date"
    ].isna().sum()
)

print(
    "Matches missing dates:",
    historical_with_metadata.loc[
        historical_with_metadata[
            "match_date"
        ].isna(),
        "match_id"
    ].nunique()
)

Merged shape: (121034, 22)
Unique matches: 4235
Player-match rows missing dates: 7564
Matches missing dates: 274


In [95]:
historical_clean = (
    historical_with_metadata
    .dropna(
        subset=["match_date"]
    )
    .copy()
)

print(
    "Clean historical shape:",
    historical_clean.shape
)

print(
    "Unique matches:",
    historical_clean[
        "match_id"
    ].nunique()
)

print(
    "Missing dates:",
    historical_clean[
        "match_date"
    ].isna().sum()
)

print(
    "Unique player IDs:",
    historical_clean[
        "player_id"
    ].nunique()
)

Clean historical shape: (113470, 22)
Unique matches: 3961
Missing dates: 0
Unique player IDs: 9884


In [96]:
historical_clean = (
    historical_clean
    .sort_values(
        by=[
            "player_id",
            "match_date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)

historical_clean[
    [
        "player_id",
        "player_name",
        "match_date",
        "match_id",
        "pass_attempts",
        "pass_completion_rate"
    ]
].head(20)

,player_id,player_name,match_date,match_id,pass_attempts,pass_completion_rate
0,2935.0,Nordi Mukiele Mulere,2022-08-06,3837650,24,83.333333
1,2935.0,Nordi Mukiele Mulere,2022-08-13,3837659,2,100.000000
2,2935.0,Nordi Mukiele Mulere,2022-08-21,3837662,1,100.000000
3,2935.0,Nordi Mukiele Mulere,2022-08-28,3837680,5,80.000000
4,2935.0,Nordi Mukiele Mulere,2022-08-31,3837683,33,93.939394
5,2935.0,Nordi Mukiele Mulere,2022-09-18,3837717,22,86.363636
6,2935.0,Nordi Mukiele Mulere,2022-10-01,3837727,79,91.139241
7,2935.0,Nordi Mukiele Mulere,2022-10-16,3837747,26,88.461538
8,2935.0,Nordi Mukiele Mulere,2022-10-21,3837752,54,90.740741
9,2935.0,Nordi Mukiele Mulere,2022-10-29,3837771,48,91.666667


In [97]:
# Previous-match pass completion
historical_clean[
    "prev_pass_completion_rate"
] = (
    historical_clean
    .groupby("player_id")[
        "pass_completion_rate"
    ]
    .shift(1)
)


# Rolling average from previous 3 matches
historical_clean[
    "rolling_3_pass_completion"
] = (
    historical_clean
    .groupby("player_id")[
        "pass_completion_rate"
    ]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()
    )
)


# Rolling average from previous 5 matches
historical_clean[
    "rolling_5_pass_completion"
] = (
    historical_clean
    .groupby("player_id")[
        "pass_completion_rate"
    ]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=5,
            min_periods=1
        )
        .mean()
    )
)

In [98]:
historical_clean[
    [
        "player_id",
        "player_name",
        "match_date",
        "pass_completion_rate",
        "prev_pass_completion_rate",
        "rolling_3_pass_completion",
        "rolling_5_pass_completion"
    ]
].head(15)

,player_id,player_name,match_date,pass_completion_rate,prev_pass_completion_rate,rolling_3_pass_completion,rolling_5_pass_completion
0,2935.0,Nordi Mukiele Mulere,2022-08-06,83.333333,NaN,NaN,NaN
1,2935.0,Nordi Mukiele Mulere,2022-08-13,100.000000,83.333333,83.333333,83.333333
2,2935.0,Nordi Mukiele Mulere,2022-08-21,100.000000,100.000000,91.666667,91.666667
3,2935.0,Nordi Mukiele Mulere,2022-08-28,80.000000,100.000000,94.444444,94.444444
4,2935.0,Nordi Mukiele Mulere,2022-08-31,93.939394,80.000000,93.333333,90.833333
5,2935.0,Nordi Mukiele Mulere,2022-09-18,86.363636,93.939394,91.313131,91.454545
6,2935.0,Nordi Mukiele Mulere,2022-10-01,91.139241,86.363636,86.767677,92.060606
7,2935.0,Nordi Mukiele Mulere,2022-10-16,88.461538,91.139241,90.480757,90.288454
8,2935.0,Nordi Mukiele Mulere,2022-10-21,90.740741,88.461538,88.654805,87.980762
9,2935.0,Nordi Mukiele Mulere,2022-10-29,91.666667,90.740741,90.113840,90.128910


In [99]:
historical_metrics = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for metric in historical_metrics:

    # Previous match
    historical_clean[
        f"prev_{metric}"
    ] = (
        historical_clean
        .groupby("player_id")[metric]
        .shift(1)
    )

    # Previous 3 matches
    historical_clean[
        f"rolling_3_{metric}"
    ] = (
        historical_clean
        .groupby("player_id")[metric]
        .transform(
            lambda x:
            x.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )

    # Previous 5 matches
    historical_clean[
        f"rolling_5_{metric}"
    ] = (
        historical_clean
        .groupby("player_id")[metric]
        .transform(
            lambda x:
            x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )

In [100]:
historical_clean[
    "previous_matches_available"
] = (
    historical_clean
    .groupby("player_id")
    .cumcount()
)

In [101]:
historical_clean[
    [
        "player_id",
        "player_name",
        "match_date",
        "previous_matches_available",
        "pass_attempts",
        "rolling_3_pass_attempts",
        "rolling_5_pass_attempts",
        "progressive_passes",
        "rolling_3_progressive_passes",
        "carries",
        "rolling_3_carries",
        "shots",
        "rolling_3_shots"
    ]
].head(15)

,player_id,player_name,match_date,previous_matches_available,pass_attempts,rolling_3_pass_attempts,rolling_5_pass_attempts,progressive_passes,rolling_3_progressive_passes,carries,rolling_3_carries,shots,rolling_3_shots
0,2935.0,Nordi Mukiele Mulere,2022-08-06,0,24,NaN,NaN,1,NaN,22,NaN,0,NaN
1,2935.0,Nordi Mukiele Mulere,2022-08-13,1,2,24.000000,24.0,0,1.000000,0,22.000000,0,0.000000
2,2935.0,Nordi Mukiele Mulere,2022-08-21,2,1,13.000000,13.0,0,0.500000,1,11.000000,0,0.000000
3,2935.0,Nordi Mukiele Mulere,2022-08-28,3,5,9.000000,9.0,1,0.333333,2,7.666667,0,0.000000
4,2935.0,Nordi Mukiele Mulere,2022-08-31,4,33,2.666667,8.0,1,0.333333,23,1.000000,1,0.000000
5,2935.0,Nordi Mukiele Mulere,2022-09-18,5,22,13.000000,13.0,3,0.666667,15,8.666667,0,0.333333
6,2935.0,Nordi Mukiele Mulere,2022-10-01,6,79,20.000000,12.6,5,1.666667,59,13.333333,1,0.333333
7,2935.0,Nordi Mukiele Mulere,2022-10-16,7,26,44.666667,28.0,3,3.000000,19,32.333333,0,0.666667
8,2935.0,Nordi Mukiele Mulere,2022-10-21,8,54,42.333333,33.0,6,3.666667,46,31.000000,0,0.333333
9,2935.0,Nordi Mukiele Mulere,2022-10-29,9,48,53.000000,42.8,3,4.666667,39,41.333333,2,0.333333


In [102]:
print("Historical:")
print(
    historical_clean[
        ["match_id", "player_id"]
    ].dtypes
)

print("\nMinutes:")
print(
    all_player_minutes[
        ["match_id", "player_id"]
    ].dtypes
)

Historical:
match_id       int64
player_id    float64
dtype: object

Minutes:
match_id     int64
player_id    int64
dtype: object


In [103]:
historical_clean["player_id"] = (
    pd.to_numeric(
        historical_clean["player_id"],
        errors="coerce"
    )
    .astype("Int64")
)

all_player_minutes["player_id"] = (
    pd.to_numeric(
        all_player_minutes["player_id"],
        errors="coerce"
    )
    .astype("Int64")
)

In [104]:
print(
    historical_clean[
        ["match_id", "player_id"]
    ].dtypes
)

print(
    all_player_minutes[
        ["match_id", "player_id"]
    ].dtypes
)

match_id     int64
player_id    Int64
dtype: object
match_id     int64
player_id    Int64
dtype: object


In [105]:
minutes_for_merge = all_player_minutes[
    [
        "match_id",
        "player_id",
        "minutes_played",
        "position_changes",
        "temporary_off_pitch_gaps",
        "match_duration_minutes"
    ]
].copy()

historical_with_minutes = historical_clean.merge(
    minutes_for_merge,
    on=[
        "match_id",
        "player_id"
    ],
    how="left",
    validate="one_to_one"
)

In [106]:
print(
    "Merged shape:",
    historical_with_minutes.shape
)

print(
    "Missing minutes:",
    historical_with_minutes[
        "minutes_played"
    ].isna().sum()
)

print(
    "Duplicate match-player rows:",
    historical_with_minutes.duplicated(
        subset=[
            "match_id",
            "player_id"
        ]
    ).sum()
)

Merged shape: (113470, 48)
Missing minutes: 0
Duplicate match-player rows: 0


In [107]:
per90_metrics = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for metric in per90_metrics:

    historical_with_minutes[
        f"{metric}_per90"
    ] = np.where(
        historical_with_minutes[
            "minutes_played"
        ] > 0,
        (
            historical_with_minutes[
                metric
            ]
            /
            historical_with_minutes[
                "minutes_played"
            ]
        ) * 90,
        np.nan
    )

In [108]:
historical_with_minutes[
    [
        "player_id",
        "player_name",
        "match_date",
        "minutes_played",
        "pass_attempts",
        "pass_attempts_per90",
        "progressive_passes",
        "progressive_passes_per90",
        "carries",
        "carries_per90",
        "shots",
        "shots_per90"
    ]
].head(20)

,player_id,player_name,match_date,minutes_played,pass_attempts,pass_attempts_per90,progressive_passes,progressive_passes_per90,carries,carries_per90,shots,shots_per90
0,2935,Nordi Mukiele Mulere,2022-08-06,23.183333,24,93.170381,1,3.882099,22,85.406183,0,0.000000
1,2935,Nordi Mukiele Mulere,2022-08-13,6.500000,2,27.692308,0,0.000000,0,0.000000,0,0.000000
2,2935,Nordi Mukiele Mulere,2022-08-21,9.450000,1,9.523810,0,0.000000,1,9.523810,0,0.000000
3,2935,Nordi Mukiele Mulere,2022-08-28,9.683333,5,46.471601,1,9.294320,2,18.588640,0,0.000000
4,2935,Nordi Mukiele Mulere,2022-08-31,92.033333,33,32.270916,1,0.977907,23,22.491851,1,0.977907
5,2935,Nordi Mukiele Mulere,2022-09-18,31.466667,22,62.923729,3,8.580508,15,42.902542,0,0.000000
6,2935,Nordi Mukiele Mulere,2022-10-01,94.083333,79,75.571302,5,4.782994,59,56.439327,1,0.956599
7,2935,Nordi Mukiele Mulere,2022-10-16,70.200000,26,33.333333,3,3.846154,19,24.358974,0,0.000000
8,2935,Nordi Mukiele Mulere,2022-10-21,92.966667,54,52.276802,6,5.808534,46,44.532090,0,0.000000
9,2935,Nordi Mukiele Mulere,2022-10-29,93.000000,48,46.451613,3,2.903226,39,37.741935,2,1.935484


In [109]:
print(
    historical_with_minutes[
        "minutes_played"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nAppearances under 5 minutes:")
print(
    (
        historical_with_minutes[
            "minutes_played"
        ] < 5
    ).sum()
)

print("\nAppearances under 10 minutes:")
print(
    (
        historical_with_minutes[
            "minutes_played"
        ] < 10
    ).sum()
)

print("\nAppearances under 20 minutes:")
print(
    (
        historical_with_minutes[
            "minutes_played"
        ] < 20
    ).sum()
)

print("\nAppearances under 30 minutes:")
print(
    (
        historical_with_minutes[
            "minutes_played"
        ] < 30
    ).sum()
)

print("\nZero-minute records:")
print(
    (
        historical_with_minutes[
            "minutes_played"
        ] == 0
    ).sum()
)

count    113470.000000
mean         72.835496
std          29.723926
min           0.000000
1%            5.511500
5%           13.216667
10%          20.766667
25%          52.083333
50%          92.066667
75%          94.133333
90%          95.950000
95%          97.216667
99%         102.350000
max         127.633333
Name: minutes_played, dtype: float64

Appearances under 5 minutes:
927

Appearances under 10 minutes:
3518

Appearances under 20 minutes:
10770

Appearances under 30 minutes:
17773

Zero-minute records:
38


In [110]:
historical_with_minutes[
    "reliable_per90"
] = (
    historical_with_minutes[
        "minutes_played"
    ] >= 20
)

print(
    historical_with_minutes[
        "reliable_per90"
    ].value_counts()
)

print(
    historical_with_minutes[
        "reliable_per90"
    ].value_counts(
        normalize=True
    ) * 100
)

reliable_per90
True     102700
False     10770
Name: count, dtype: int64
reliable_per90
True     90.508504
False     9.491496
Name: proportion, dtype: float64


In [111]:
historical_with_minutes.loc[
    historical_with_minutes[
        "minutes_played"
    ] == 0,
    [
        "match_id",
        "player_id",
        "player_name",
        "match_date",
        "total_events",
        "pass_attempts",
        "shots",
        "carries",
        "minutes_played"
    ]
].head(40)

,match_id,player_id,player_name,match_date,total_events,pass_attempts,shots,carries,minutes_played
7239,3920413,3417,Saidy Janko,2024-01-23,1,0,0,0,0.0
23267,3773497,5213,Gerard Piqué Bernabéu,2021-04-10,1,0,0,0,0.0
29996,3857259,6319,Luka Jović,2022-11-28,1,0,0,0,0.0
33860,3825869,6672,Jorge Andújar Moreno,2016-05-14,1,0,0,0,0.0
35723,303615,6758,Víctor Sánchez Mata,2020-07-08,1,0,0,0,0.0
40910,3930184,7044,Patrik Schick,2024-06-26,1,0,0,0,0.0
41363,3825743,7105,Fabián Ariel Orellana Valenzuela,2016-01-23,1,0,0,0,0.0
52864,3912527,10371,Elisa Bartoli,2024-01-20,1,0,0,0,0.0
53529,3900528,10448,Adam Ounas,2015-11-29,1,0,0,0,0.0
53929,3825885,10612,Yoel Rodríguez Oterino,2016-04-01,1,0,0,0,0.0


In [112]:
match_id = 3893806

lineup_file = Path(
    f"open-data-master/data/lineups/{match_id}.json"
)

with open(
    lineup_file,
    "r",
    encoding="utf-8"
) as f:
    lineup_data = json.load(f)

for team in lineup_data:
    for player in team["lineup"]:

        if player.get("player_id") == 49913:

            print(
                "Team:",
                team["team_name"]
            )

            print(
                "Player:",
                player["player_name"]
            )

            print("\nPositions:")

            for position in player.get(
                "positions",
                []
            ):
                print(position)

Team: Spain Women's
Player: Athenea del Castillo Belvide

Positions:
{'position_id': 21, 'position': 'Left Wing', 'from': '100:16', 'to': '96:36', 'from_period': 2, 'to_period': 2, 'start_reason': 'Player On', 'end_reason': 'Player Off'}


In [113]:
reversed_intervals = []

lineups_folder = Path(
    "open-data-master/data/lineups"
)

for lineup_file in lineups_folder.glob("*.json"):

    match_id = int(lineup_file.stem)

    with open(
        lineup_file,
        "r",
        encoding="utf-8"
    ) as f:
        lineup_data = json.load(f)

    for team in lineup_data:

        for player in team["lineup"]:

            for position in player.get(
                "positions",
                []
            ):

                start_time = position.get("from")
                end_time = position.get("to")

                if (
                    pd.notna(start_time)
                    and pd.notna(end_time)
                ):

                    start_seconds = clock_to_seconds(
                        start_time
                    )

                    end_seconds = clock_to_seconds(
                        end_time
                    )

                    if end_seconds < start_seconds:

                        reversed_intervals.append({
                            "match_id": match_id,
                            "player_id":
                                player.get("player_id"),
                            "player_name":
                                player.get("player_name"),
                            "position":
                                position.get("position"),
                            "from": start_time,
                            "to": end_time,
                            "start_reason":
                                position.get("start_reason"),
                            "end_reason":
                                position.get("end_reason")
                        })

reversed_intervals_df = pd.DataFrame(
    reversed_intervals
)

print(
    "Reversed intervals:",
    len(reversed_intervals_df)
)

print(
    "Affected matches:",
    reversed_intervals_df[
        "match_id"
    ].nunique()
)

print(
    "Affected players:",
    reversed_intervals_df[
        "player_id"
    ].nunique()
)

reversed_intervals_df.head(30)

Reversed intervals: 183
Affected matches: 43
Affected players: 166


,match_id,player_id,player_name,position,from,to,start_reason,end_reason
0,2302764,15280,Vladimir Smicer,Right Midfield,105:38,22:33,Player On,Substitution - On (Tactical)
1,3794685,7156,Federico Chiesa,Left Wing,108:54,83:34,Tactical Shift,Substitution - On (Tactical)
2,3794692,8552,Robin Quaison,Center Forward,109:28,96:08,Tactical Shift,Substitution - On (Tactical)
3,3794692,11098,Marcus Berg,Right Defensive Midfield,109:28,96:38,Tactical Shift,Substitution - On (Tactical)
4,3794692,26875,Marcus Andreas Danielsson,Right Center Midfield,109:28,97:04,Tactical Shift,Foul Committed (Red Card)
5,3795108,3957,César Azpilicueta Tanco,Right Center Back,113:03,28:11,Tactical Shift,Tactical Shift
6,3795108,4353,Aymeric Laporte,Left Center Back,113:03,28:11,Tactical Shift,Tactical Shift
7,3795108,6685,Mikel Oyarzabal Ugarte,Right Wing,113:03,90:00,Tactical Shift,Substitution - On (Tactical)
8,3795108,6766,Gerard Moreno Balaguero,Center Forward,113:03,76:20,Tactical Shift,Tactical Shift
9,3795108,6840,Marcos Llorente Moreno,Right Back,113:03,90:01,Tactical Shift,Tactical Shift


In [114]:
print(
    "Reversed intervals:",
    len(reversed_intervals_df)
)

print(
    "Affected matches:",
    reversed_intervals_df[
        "match_id"
    ].nunique()
)

print(
    "Affected players:",
    reversed_intervals_df[
        "player_id"
    ].nunique()
)

Reversed intervals: 183
Affected matches: 43
Affected players: 166


In [115]:
reversed_player_matches = (
    reversed_intervals_df[
        [
            "match_id",
            "player_id"
        ]
    ]
    .drop_duplicates()
    .copy()
)

reversed_player_matches[
    "invalid_minutes_interval"
] = True

print(
    "Affected player-match records:",
    len(reversed_player_matches)
)

Affected player-match records: 183


In [116]:
reversed_player_matches[
    "player_id"
] = (
    pd.to_numeric(
        reversed_player_matches[
            "player_id"
        ],
        errors="coerce"
    )
    .astype("Int64")
)

In [117]:
historical_with_minutes = (
    historical_with_minutes.merge(
        reversed_player_matches,
        on=[
            "match_id",
            "player_id"
        ],
        how="left",
        validate="one_to_one"
    )
)

historical_with_minutes[
    "invalid_minutes_interval"
] = (
    historical_with_minutes[
        "invalid_minutes_interval"
    ]
    .fillna(False)
    .astype(bool)
)

In [118]:
print(
    "Historical rows with invalid intervals:",
    historical_with_minutes[
        "invalid_minutes_interval"
    ].sum()
)

print(
    "Invalid intervals with Events activity:",
    (
        historical_with_minutes.loc[
            historical_with_minutes[
                "invalid_minutes_interval"
            ],
            "total_events"
        ] > 0
    ).sum()
)

Historical rows with invalid intervals: 183
Invalid intervals with Events activity: 183


In [119]:
historical_with_minutes[
    "reliable_per90"
] = (
    (
        historical_with_minutes[
            "minutes_played"
        ] >= 20
    )
    &
    (
        ~historical_with_minutes[
            "invalid_minutes_interval"
        ]
    )
)

print(
    historical_with_minutes[
        "reliable_per90"
    ].value_counts()
)

print("\nPercentages:")

print(
    historical_with_minutes[
        "reliable_per90"
    ].value_counts(
        normalize=True
    ) * 100
)

reliable_per90
True     102526
False     10944
Name: count, dtype: int64

Percentages:
reliable_per90
True     90.35516
False     9.64484
Name: proportion, dtype: float64


In [120]:
per90_metrics = [
    "pass_attempts_per90",
    "progressive_passes_per90",
    "carries_per90",
    "shots_per90",
    "miscontrols_per90",
    "shot_assists_per90"
]

for metric in per90_metrics:

    historical_with_minutes[
        f"reliable_{metric}"
    ] = historical_with_minutes[
        metric
    ].where(
        historical_with_minutes[
            "reliable_per90"
        ]
    )

In [121]:
historical_with_minutes = (
    historical_with_minutes
    .sort_values(
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)

In [122]:
for metric in per90_metrics:

    reliable_metric = (
        f"reliable_{metric}"
    )

    historical_with_minutes[
        f"prev_{metric}"
    ] = (
        historical_with_minutes
        .groupby("player_id")[
            reliable_metric
        ]
        .shift(1)
    )

    historical_with_minutes[
        f"rolling_3_{metric}"
    ] = (
        historical_with_minutes
        .groupby("player_id")[
            reliable_metric
        ]
        .transform(
            lambda x:
            x.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )

    historical_with_minutes[
        f"rolling_5_{metric}"
    ] = (
        historical_with_minutes
        .groupby("player_id")[
            reliable_metric
        ]
        .transform(
            lambda x:
            x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )

In [123]:
historical_with_minutes.loc[
    historical_with_minutes[
        "player_id"
    ] == 2935,
    [
        "match_date",
        "minutes_played",
        "reliable_per90",
        "pass_attempts_per90",
        "reliable_pass_attempts_per90",
        "prev_pass_attempts_per90",
        "rolling_3_pass_attempts_per90",
        "rolling_5_pass_attempts_per90"
    ]
].head(15)


,match_date,minutes_played,reliable_per90,pass_attempts_per90,reliable_pass_attempts_per90,prev_pass_attempts_per90,rolling_3_pass_attempts_per90,rolling_5_pass_attempts_per90
0,2022-08-06,23.183333,True,93.170381,93.170381,NaN,NaN,NaN
1,2022-08-13,6.500000,False,27.692308,NaN,93.170381,93.170381,93.170381
2,2022-08-21,9.450000,False,9.523810,NaN,NaN,93.170381,93.170381
3,2022-08-28,9.683333,False,46.471601,NaN,NaN,93.170381,93.170381
4,2022-08-31,92.033333,True,32.270916,32.270916,NaN,NaN,93.170381
5,2022-09-18,31.466667,True,62.923729,62.923729,32.270916,32.270916,62.720649
6,2022-10-01,94.083333,True,75.571302,75.571302,62.923729,47.597323,47.597323
7,2022-10-16,70.200000,True,33.333333,33.333333,75.571302,56.921982,56.921982
8,2022-10-21,92.966667,True,52.276802,52.276802,33.333333,57.276121,51.024820
9,2022-10-29,93.000000,True,46.451613,46.451613,52.276802,53.727146,51.275216


In [124]:
reliable_pass_history = (
    historical_with_minutes.loc[
        historical_with_minutes[
            "reliable_per90"
        ],
        [
            "player_id",
            "match_date",
            "match_id",
            "pass_attempts_per90"
        ]
    ]
    .copy()
)

reliable_pass_history = (
    reliable_pass_history
    .sort_values(
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    )
)

In [125]:
reliable_pass_history[
    "last_3_reliable_pass_attempts_per90"
] = (
    reliable_pass_history
    .groupby("player_id")[
        "pass_attempts_per90"
    ]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()
    )
)

reliable_pass_history[
    "last_5_reliable_pass_attempts_per90"
] = (
    reliable_pass_history
    .groupby("player_id")[
        "pass_attempts_per90"
    ]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=5,
            min_periods=1
        )
        .mean()
    )
)

In [126]:
reliable_pass_history.loc[
    reliable_pass_history[
        "player_id"
    ] == 2935
].head(10)

,player_id,match_date,match_id,pass_attempts_per90,last_3_reliable_pass_attempts_per90,last_5_reliable_pass_attempts_per90
0,2935,2022-08-06,3837650,93.170381,NaN,NaN
4,2935,2022-08-31,3837683,32.270916,93.170381,93.170381
5,2935,2022-09-18,3837717,62.923729,62.720649,62.720649
6,2935,2022-10-01,3837727,75.571302,62.788342,62.788342
7,2935,2022-10-16,3837747,33.333333,56.921982,65.984082
8,2935,2022-10-21,3837752,52.276802,57.276121,59.453932
9,2935,2022-10-29,3837771,46.451613,53.727146,51.275216
10,2935,2022-11-13,3837785,55.364335,44.020583,54.111356
11,2935,2023-01-11,3837818,64.853409,51.364250,52.599477
12,2935,2023-01-15,3837827,57.196262,55.556452,50.455898


In [127]:
pass_history_state = (
    reliable_pass_history[
        [
            "player_id",
            "match_date",
            "match_id",
            "pass_attempts_per90"
        ]
    ]
    .copy()
)

pass_history_state = (
    pass_history_state
    .sort_values(
        [
            "match_date",
            "player_id",
            "match_id"
        ]
    )
)

In [128]:
all_match_states = (
    historical_with_minutes[
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    ]
    .copy()
    .sort_values(
        [
            "match_date",
            "player_id",
            "match_id"
        ]
    )
)

In [129]:
pass_baseline_test = pd.merge_asof(
    all_match_states,
    pass_history_state,
    on="match_date",
    by="player_id",
    direction="backward",
    allow_exact_matches=False
)

In [130]:
pass_baseline_test.loc[
    pass_baseline_test[
        "player_id"
    ] == 2935
].head(15)

,player_id,match_date,match_id_x,match_id_y,pass_attempts_per90
78746,2935,2022-08-06,3837650,NaN,NaN
78777,2935,2022-08-13,3837659,3837650.0,93.170381
78809,2935,2022-08-21,3837662,3837650.0,93.170381
78840,2935,2022-08-28,3837680,3837650.0,93.170381
78869,2935,2022-08-31,3837683,3837650.0,93.170381
78965,2935,2022-09-18,3837717,3837683.0,32.270916
78994,2935,2022-10-01,3837727,3837717.0,62.923729
79026,2935,2022-10-16,3837747,3837727.0,75.571302
79056,2935,2022-10-21,3837752,3837747.0,33.333333
79088,2935,2022-10-29,3837771,3837752.0,52.276802


In [131]:
reliable_pass_history = (
    historical_with_minutes.loc[
        historical_with_minutes[
            "reliable_per90"
        ],
        [
            "player_id",
            "match_date",
            "match_id",
            "pass_attempts_per90"
        ]
    ]
    .copy()
    .sort_values(
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    )
)

In [132]:
reliable_pass_history[
    "last_3_reliable_pass_attempts_per90"
] = (
    reliable_pass_history
    .groupby("player_id")[
        "pass_attempts_per90"
    ]
    .transform(
        lambda x:
        x.rolling(
            window=3,
            min_periods=1
        ).mean()
    )
)

reliable_pass_history[
    "last_5_reliable_pass_attempts_per90"
] = (
    reliable_pass_history
    .groupby("player_id")[
        "pass_attempts_per90"
    ]
    .transform(
        lambda x:
        x.rolling(
            window=5,
            min_periods=1
        ).mean()
    )
)

In [133]:
pass_history_state = (
    reliable_pass_history[
        [
            "player_id",
            "match_date",
            "match_id",
            "pass_attempts_per90",
            "last_3_reliable_pass_attempts_per90",
            "last_5_reliable_pass_attempts_per90"
        ]
    ]
    .rename(
        columns={
            "match_id":
                "previous_reliable_match_id",
            "pass_attempts_per90":
                "previous_reliable_pass_attempts_per90"
        }
    )
    .sort_values(
        [
            "match_date",
            "player_id",
            "previous_reliable_match_id"
        ]
    )
)

In [134]:
all_match_states = (
    historical_with_minutes[
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    ]
    .copy()
    .sort_values(
        [
            "match_date",
            "player_id",
            "match_id"
        ]
    )
)

pass_baseline_test = pd.merge_asof(
    all_match_states,
    pass_history_state,
    on="match_date",
    by="player_id",
    direction="backward",
    allow_exact_matches=False
)

In [135]:
pass_baseline_test.loc[
    pass_baseline_test[
        "player_id"
    ] == 2935,
    [
        "match_date",
        "match_id",
        "previous_reliable_match_id",
        "previous_reliable_pass_attempts_per90",
        "last_3_reliable_pass_attempts_per90",
        "last_5_reliable_pass_attempts_per90"
    ]
].head(15)

,match_date,match_id,previous_reliable_match_id,previous_reliable_pass_attempts_per90,last_3_reliable_pass_attempts_per90,last_5_reliable_pass_attempts_per90
78746,2022-08-06,3837650,NaN,NaN,NaN,NaN
78777,2022-08-13,3837659,3837650.0,93.170381,93.170381,93.170381
78809,2022-08-21,3837662,3837650.0,93.170381,93.170381,93.170381
78840,2022-08-28,3837680,3837650.0,93.170381,93.170381,93.170381
78869,2022-08-31,3837683,3837650.0,93.170381,93.170381,93.170381
78965,2022-09-18,3837717,3837683.0,32.270916,62.720649,62.720649
78994,2022-10-01,3837727,3837717.0,62.923729,62.788342,62.788342
79026,2022-10-16,3837747,3837727.0,75.571302,56.921982,65.984082
79056,2022-10-21,3837752,3837747.0,33.333333,57.276121,59.453932
79088,2022-10-29,3837771,3837752.0,52.276802,53.727146,51.275216


In [136]:
baseline_metrics = [
    "pass_attempts_per90",
    "progressive_passes_per90",
    "carries_per90",
    "shots_per90",
    "miscontrols_per90",
    "shot_assists_per90"
]

reliable_history = (
    historical_with_minutes.loc[
        historical_with_minutes[
            "reliable_per90"
        ],
        [
            "player_id",
            "match_date",
            "match_id"
        ] + baseline_metrics
    ]
    .copy()
    .sort_values(
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    )
)

for metric in baseline_metrics:

    reliable_history[
        f"last_3_reliable_{metric}"
    ] = (
        reliable_history
        .groupby("player_id")[metric]
        .transform(
            lambda x:
            x.rolling(
                window=3,
                min_periods=1
            ).mean()
        )
    )

    reliable_history[
        f"last_5_reliable_{metric}"
    ] = (
        reliable_history
        .groupby("player_id")[metric]
        .transform(
            lambda x:
            x.rolling(
                window=5,
                min_periods=1
            ).mean()
        )
    )

In [137]:
reliable_history = (
    reliable_history.rename(
        columns={
            "match_id":
                "previous_reliable_match_id",
            **{
                metric:
                f"previous_reliable_{metric}"
                for metric in baseline_metrics
            }
        }
    )
)

In [138]:
current_matches = (
    historical_with_minutes[
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    ]
    .copy()
)

In [139]:
current_matches = (
    current_matches
    .sort_values(
        [
            "match_date",
            "player_id",
            "match_id"
        ]
    )
)

reliable_history = (
    reliable_history
    .sort_values(
        [
            "match_date",
            "player_id",
            "previous_reliable_match_id"
        ]
    )
)

In [140]:
historical_baselines = pd.merge_asof(
    current_matches,
    reliable_history,
    on="match_date",
    by="player_id",
    direction="backward",
    allow_exact_matches=False
)

In [141]:
print(
    "Baseline shape:",
    historical_baselines.shape
)

print(
    "Unique match-player rows:",
    historical_baselines[
        ["match_id", "player_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Rows with previous reliable history:",
    historical_baselines[
        "previous_reliable_match_id"
    ].notna().sum()
)

print(
    "Rows without previous reliable history:",
    historical_baselines[
        "previous_reliable_match_id"
    ].isna().sum()
)

Baseline shape: (113470, 22)
Unique match-player rows: 113470
Rows with previous reliable history: 101861
Rows without previous reliable history: 11609


In [142]:
historical_final = historical_with_minutes.merge(
    historical_baselines,
    on=[
        "match_id",
        "player_id",
        "match_date"
    ],
    how="left",
    validate="one_to_one"
)

In [143]:
print(
    "Final shape:",
    historical_final.shape
)

print(
    "Duplicate match-player rows:",
    historical_final.duplicated(
        subset=[
            "match_id",
            "player_id"
        ]
    ).sum()
)

print(
    "Missing baseline match IDs:",
    historical_final[
        "previous_reliable_match_id"
    ].isna().sum()
)

Final shape: (113470, 99)
Duplicate match-player rows: 0
Missing baseline match IDs: 11609


In [144]:
for i, column in enumerate(
    historical_final.columns,
    start=1
):
    print(
        i,
        column
    )

1 match_id
2 player_id
3 player_name
4 total_events
5 pass_attempts
6 passes_completed
7 forward_passes
8 progressive_passes
9 avg_pass_length
10 avg_pass_angle_degrees
11 crosses
12 through_balls
13 switches
14 shot_assists
15 goal_assists
16 shots
17 carries
18 miscontrols
19 pass_completion_rate
20 match_date
21 home_team_name
22 away_team_name
23 prev_pass_completion_rate
24 rolling_3_pass_completion
25 rolling_5_pass_completion
26 prev_pass_attempts
27 rolling_3_pass_attempts
28 rolling_5_pass_attempts
29 prev_progressive_passes
30 rolling_3_progressive_passes
31 rolling_5_progressive_passes
32 prev_carries
33 rolling_3_carries
34 rolling_5_carries
35 prev_shots
36 rolling_3_shots
37 rolling_5_shots
38 prev_miscontrols
39 rolling_3_miscontrols
40 rolling_5_miscontrols
41 prev_shot_assists
42 rolling_3_shot_assists
43 rolling_5_shot_assists
44 previous_matches_available
45 minutes_played
46 position_changes
47 temporary_off_pitch_gaps
48 match_duration_minutes
49 pass_attempts_per9

In [145]:
column_groups = {
    "identifiers_metadata": [],
    "current_performance": [],
    "historical_raw": [],
    "playing_time_quality": [],
    "per90_current": [],
    "reliable_helper": [],
    "historical_per90_baseline": [],
    "other": []
}

for col in historical_final.columns:

    if col in [
        "match_id",
        "player_id",
        "player_name",
        "match_date",
        "home_team_name",
        "away_team_name"
    ]:
        column_groups[
            "identifiers_metadata"
        ].append(col)

    elif col in [
        "minutes_played",
        "match_duration_minutes",
        "position_changes",
        "temporary_off_pitch_gaps",
        "invalid_minutes_interval",
        "reliable_per90"
    ]:
        column_groups[
            "playing_time_quality"
        ].append(col)

    elif col.startswith(
        (
            "previous_reliable_",
            "last_3_reliable_",
            "last_5_reliable_"
        )
    ):
        column_groups[
            "historical_per90_baseline"
        ].append(col)

    elif col.startswith("reliable_"):
        column_groups[
            "reliable_helper"
        ].append(col)

    elif col.endswith("_per90"):
        column_groups[
            "per90_current"
        ].append(col)

    elif col.startswith(
        (
            "prev_",
            "rolling_3_",
            "rolling_5_"
        )
    ) or col == "previous_matches_available":
        column_groups[
            "historical_raw"
        ].append(col)

    elif col in [
        "total_events",
        "pass_attempts",
        "passes_completed",
        "forward_passes",
        "progressive_passes",
        "avg_pass_length",
        "avg_pass_angle_degrees",
        "crosses",
        "through_balls",
        "switches",
        "shot_assists",
        "goal_assists",
        "shots",
        "carries",
        "miscontrols",
        "pass_completion_rate"
    ]:
        column_groups[
            "current_performance"
        ].append(col)

    else:
        column_groups[
            "other"
        ].append(col)


for group, columns in column_groups.items():

    print(
        f"\n{group.upper()} "
        f"({len(columns)} columns)"
    )

    print(columns)


IDENTIFIERS_METADATA (6 columns)
['match_id', 'player_id', 'player_name', 'match_date', 'home_team_name', 'away_team_name']

CURRENT_PERFORMANCE (16 columns)
['total_events', 'pass_attempts', 'passes_completed', 'forward_passes', 'progressive_passes', 'avg_pass_length', 'avg_pass_angle_degrees', 'crosses', 'through_balls', 'switches', 'shot_assists', 'goal_assists', 'shots', 'carries', 'miscontrols', 'pass_completion_rate']

HISTORICAL_RAW (22 columns)
['prev_pass_completion_rate', 'rolling_3_pass_completion', 'rolling_5_pass_completion', 'prev_pass_attempts', 'rolling_3_pass_attempts', 'rolling_5_pass_attempts', 'prev_progressive_passes', 'rolling_3_progressive_passes', 'rolling_5_progressive_passes', 'prev_carries', 'rolling_3_carries', 'rolling_5_carries', 'prev_shots', 'rolling_3_shots', 'rolling_5_shots', 'prev_miscontrols', 'rolling_3_miscontrols', 'rolling_5_miscontrols', 'prev_shot_assists', 'rolling_3_shot_assists', 'rolling_5_shot_assists', 'previous_matches_available']

PLA

In [146]:
historical_model_features = historical_final.copy()

old_per90_history_columns = [
    col
    for col in historical_model_features.columns
    if (
        col.startswith("prev_")
        or col.startswith("rolling_3_")
        or col.startswith("rolling_5_")
    )
    and col.endswith("_per90")
]

historical_model_features = (
    historical_model_features.drop(
        columns=old_per90_history_columns
    )
)

print(
    "Removed old per-90 historical columns:",
    len(old_per90_history_columns)
)

print(
    "New shape:",
    historical_model_features.shape
)

Removed old per-90 historical columns: 18
New shape: (113470, 81)


In [147]:
historical_final.to_pickle(
    "historical_player_match_master.pkl"
)

historical_model_features.to_pickle(
    "historical_player_match_features.pkl"
)

print("Saved successfully.")

Saved successfully.


In [148]:
import math

def get_nearest_teammate(freeze_frame):

    if not isinstance(freeze_frame, list):
        return None, None

    actor = next(
        (
            player
            for player in freeze_frame
            if player.get("actor") == True
            and isinstance(
                player.get("location"),
                list
            )
        ),
        None
    )

    if actor is None:
        return None, None

    actor_location = actor["location"]

    teammates = [
        player
        for player in freeze_frame
        if player.get("teammate") == True
        and player.get("actor") != True
        and isinstance(
            player.get("location"),
            list
        )
    ]

    if len(teammates) == 0:
        return None, None

    distances = []

    for teammate in teammates:

        teammate_location = teammate["location"]

        distance = math.sqrt(
            (
                teammate_location[0]
                - actor_location[0]
            ) ** 2
            +
            (
                teammate_location[1]
                - actor_location[1]
            ) ** 2
        )

        distances.append(distance)

    nearest_index = distances.index(
        min(distances)
    )

    nearest_teammate = teammates[
        nearest_index
    ]

    return (
        nearest_teammate["location"],
        distances[nearest_index]
    )

In [150]:
import pandas as pd

all_360_events = pd.read_pickle(
    "all_360_prepared_events.pkl"
)

print(all_360_events.shape)
print(all_360_events["match_id"].nunique())

(1582258, 52)
425


In [151]:
test_match = (
    all_360_events[
        all_360_events["match_id"] == 3764440
    ]
    .copy()
)

print(test_match.shape)

(0, 52)


In [152]:
print(
    "freeze_frame" in test_match.columns
)

print(
    "Usable freeze frames:",
    test_match["freeze_frame"].notna().sum()
)

True
Usable freeze frames: 0


In [154]:
print(
    test_match[
        [
            "match_id",
            "id",
            "freeze_frame"
        ]
    ].head(10)
)

Empty DataFrame
Columns: [match_id, id, freeze_frame]
Index: []


In [155]:
print(
    test_match[
        "freeze_frame"
    ].apply(type).value_counts()
)

Series([], Name: count, dtype: int64)


In [156]:
print(
    test_match.columns.tolist()
)

['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession', 'possession_team', 'play_pattern', 'team', 'duration', 'tactics', 'related_events', 'player', 'position', 'location', 'pass', 'carry', 'under_pressure', 'dribble', 'shot', 'goalkeeper', 'out', 'ball_recovery', 'duel', 'ball_receipt', 'clearance', 'off_camera', 'counterpress', 'interception', 'foul_won', 'foul_committed', 'substitution', '50_50', 'injury_stoppage', 'event_uuid', 'freeze_frame', 'visible_area', 'match_id', 'event_type', 'player_name', 'nearest_opponent_location', 'nearest_opponent_distance', 'nearby_opponents_5', 'under_spatial_pressure', 'pass_completed', 'block', 'bad_behaviour', 'miscontrol', 'player_off', 'half_start']


In [157]:
print(
    "Shape:",
    all_360_events.shape
)

print(
    "Unique matches:",
    all_360_events["match_id"].nunique()
)

print(
    "match_id dtype:",
    all_360_events["match_id"].dtype
)

print(
    "First 20 match IDs:"
)

print(
    all_360_events["match_id"]
    .dropna()
    .unique()[:20]
)

print(
    "3764440 present:",
    all_360_events["match_id"]
    .astype(str)
    .eq("3764440")
    .any()
)

Shape: (1582258, 52)
Unique matches: 425
match_id dtype: str
First 20 match IDs:
<StringArray>
['3764440', '3764661', '3773369', '3773372', '3773377', '3773386', '3773387',
 '3773403', '3773415', '3773428', '3773457', '3773466', '3773474', '3773477',
 '3773497', '3773523', '3773526', '3773547', '3773552', '3773565']
Length: 20, dtype: str
3764440 present: True


In [158]:
test_match = (
    all_360_events[
        all_360_events["match_id"] == "3764440"
    ]
    .copy()
)

print(
    "Test match shape:",
    test_match.shape
)

print(
    "Usable freeze frames:",
    test_match["freeze_frame"].notna().sum()
)

Test match shape: (4160, 52)
Usable freeze frames: 3914


In [159]:
test_freeze_frame = (
    test_match["freeze_frame"]
    .dropna()
    .iloc[0]
)

nearest_teammate_location, nearest_teammate_distance = (
    get_nearest_teammate(
        test_freeze_frame
    )
)

print(
    "Nearest teammate location:",
    nearest_teammate_location
)

print(
    "Nearest teammate distance:",
    nearest_teammate_distance
)

Nearest teammate location: [61.76994556042512, 30.461066072962247]
Nearest teammate distance: 9.669634636330796


In [160]:
def count_nearby_teammates(
    freeze_frame,
    radius=5
):

    if not isinstance(
        freeze_frame,
        list
    ):
        return None

    actor = next(
        (
            player
            for player in freeze_frame
            if player.get("actor") == True
            and isinstance(
                player.get("location"),
                list
            )
        ),
        None
    )

    if actor is None:
        return None

    actor_location = actor["location"]

    count = 0

    for player in freeze_frame:

        if (
            player.get("teammate") == True
            and player.get("actor") != True
            and isinstance(
                player.get("location"),
                list
            )
        ):

            teammate_location = (
                player["location"]
            )

            distance = math.sqrt(
                (
                    teammate_location[0]
                    - actor_location[0]
                ) ** 2
                +
                (
                    teammate_location[1]
                    - actor_location[1]
                ) ** 2
            )

            if distance <= radius:
                count += 1

    return count

In [161]:
nearby_teammates_5 = (
    count_nearby_teammates(
        test_freeze_frame,
        radius=5
    )
)

print(
    "Nearby teammates within 5 units:",
    nearby_teammates_5
)

Nearby teammates within 5 units: 0


In [162]:
def calculate_local_numerical_balance(
    freeze_frame,
    radius=5
):

    nearby_teammates = count_nearby_teammates(
        freeze_frame,
        radius=radius
    )

    nearby_opponents = count_nearby_opponents(
        freeze_frame,
        radius=radius
    )

    if (
        nearby_teammates is None
        or nearby_opponents is None
    ):
        return None

    return (
        nearby_teammates
        - nearby_opponents
    )

In [164]:
import math

def count_nearby_opponents(
    freeze_frame,
    radius=5
):

    if not isinstance(
        freeze_frame,
        list
    ):
        return None

    actor = next(
        (
            player
            for player in freeze_frame
            if player.get("actor") == True
            and isinstance(
                player.get("location"),
                list
            )
        ),
        None
    )

    if actor is None:
        return None

    actor_location = actor["location"]

    count = 0

    for player in freeze_frame:

        if (
            player.get("teammate") == False
            and isinstance(
                player.get("location"),
                list
            )
        ):

            opponent_location = (
                player["location"]
            )

            distance = math.sqrt(
                (
                    opponent_location[0]
                    - actor_location[0]
                ) ** 2
                +
                (
                    opponent_location[1]
                    - actor_location[1]
                ) ** 2
            )

            if distance <= radius:
                count += 1

    return count

In [165]:
nearby_teammates_5 = (
    count_nearby_teammates(
        test_freeze_frame,
        radius=5
    )
)

nearby_opponents_5 = (
    count_nearby_opponents(
        test_freeze_frame,
        radius=5
    )
)

print(
    "Nearby teammates:",
    nearby_teammates_5
)

print(
    "Nearby opponents:",
    nearby_opponents_5
)

Nearby teammates: 0
Nearby opponents: 0


In [166]:
local_balance_5 = (
    calculate_local_numerical_balance(
        test_freeze_frame,
        radius=5
    )
)

print(
    "Local numerical balance within 5 units:",
    local_balance_5
)

Local numerical balance within 5 units: 0


In [167]:
test_match[
    "nearest_teammate_location"
] = test_match["freeze_frame"].apply(
    lambda x: get_nearest_teammate(x)[0]
)

test_match[
    "nearest_teammate_distance"
] = test_match["freeze_frame"].apply(
    lambda x: get_nearest_teammate(x)[1]
)

test_match[
    "nearby_teammates_5"
] = test_match["freeze_frame"].apply(
    lambda x: count_nearby_teammates(
        x,
        radius=5
    )
)

test_match[
    "local_numerical_balance_5"
] = (
    test_match["nearby_teammates_5"]
    -
    test_match["nearby_opponents_5"]
)

In [168]:
print(
    test_match[
        [
            "nearest_opponent_distance",
            "nearest_teammate_distance",
            "nearby_opponents_5",
            "nearby_teammates_5",
            "local_numerical_balance_5"
        ]
    ].describe()
)

       nearest_teammate_distance  nearby_teammates_5
count                3914.000000         3914.000000
mean                   10.462148            0.144609
std                     4.820214            0.418139
min                     0.015555            0.000000
25%                     6.972747            0.000000
50%                     9.919684            0.000000
75%                    13.356571            0.000000
max                    46.725621            5.000000


In [169]:
context_columns = [
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "local_numerical_balance_5"
]

print(
    test_match[
        context_columns
    ].isna().sum()
)

nearest_opponent_distance    246
nearest_teammate_distance    246
nearby_opponents_5           246
nearby_teammates_5           246
local_numerical_balance_5    246
dtype: int64


In [170]:
print(
    test_match[
        [
            "nearest_opponent_distance",
            "nearest_teammate_distance",
            "nearby_opponents_5",
            "nearby_teammates_5",
            "local_numerical_balance_5"
        ]
    ].describe()
)

       nearest_teammate_distance  nearby_teammates_5
count                3914.000000         3914.000000
mean                   10.462148            0.144609
std                     4.820214            0.418139
min                     0.015555            0.000000
25%                     6.972747            0.000000
50%                     9.919684            0.000000
75%                    13.356571            0.000000
max                    46.725621            5.000000


In [171]:
test_match[
    "nearby_teammates_10"
] = test_match["freeze_frame"].apply(
    lambda x: count_nearby_teammates(
        x,
        radius=10
    )
)

test_match[
    "nearby_opponents_10"
] = test_match["freeze_frame"].apply(
    lambda x: count_nearby_opponents(
        x,
        radius=10
    )
)

test_match[
    "local_numerical_balance_10"
] = (
    test_match["nearby_teammates_10"]
    -
    test_match["nearby_opponents_10"]
)

In [172]:
print(
    test_match[
        [
            "nearby_teammates_5",
            "nearby_teammates_10",
            "nearby_opponents_5",
            "nearby_opponents_10",
            "local_numerical_balance_5",
            "local_numerical_balance_10"
        ]
    ].describe()
)

       nearby_teammates_5  nearby_teammates_10  nearby_opponents_10  \
count         3914.000000          3914.000000          3914.000000   
mean             0.144609             0.804803             1.615994   
std              0.418139             1.060509             1.309487   
min              0.000000             0.000000             0.000000   
25%              0.000000             0.000000             1.000000   
50%              0.000000             1.000000             1.000000   
75%              0.000000             1.000000             2.000000   
max              5.000000             7.000000             8.000000   

       local_numerical_balance_10  
count                 3914.000000  
mean                    -0.811191  
std                      1.245638  
min                     -6.000000  
25%                     -1.000000  
50%                     -1.000000  
75%                      0.000000  
max                      4.000000  


In [173]:
print(
    "Teammate 5-unit zeros:",
    (test_match["nearby_teammates_5"] == 0).sum()
)

print(
    "Teammate 10-unit zeros:",
    (test_match["nearby_teammates_10"] == 0).sum()
)

print(
    "Opponent 5-unit zeros:",
    (test_match["nearby_opponents_5"] == 0).sum()
)

print(
    "Opponent 10-unit zeros:",
    (test_match["nearby_opponents_10"] == 0).sum()
)

Teammate 5-unit zeros: 3426
Teammate 10-unit zeros: 1937
Opponent 5-unit zeros: 1913
Opponent 10-unit zeros: 828


In [174]:
print(
    test_match[
        [
            "local_numerical_balance_5",
            "local_numerical_balance_10"
        ]
    ].describe()
)

       local_numerical_balance_10
count                 3914.000000
mean                    -0.811191
std                      1.245638
min                     -6.000000
25%                     -1.000000
50%                     -1.000000
75%                      0.000000
max                      4.000000


In [175]:
print(
    "5-unit balance frequencies:"
)

print(
    test_match[
        "local_numerical_balance_5"
    ].value_counts(
        dropna=False
    ).sort_index()
)

print(
    "\n10-unit balance frequencies:"
)

print(
    test_match[
        "local_numerical_balance_10"
    ].value_counts(
        dropna=False
    ).sort_index()
)

5-unit balance frequencies:
local_numerical_balance_5
-6.0       2
-4.0       5
-3.0      52
-2.0     298
-1.0    1366
 0.0    2085
 1.0      99
 2.0       3
 3.0       3
 4.0       1
 NaN     246
Name: count, dtype: int64

10-unit balance frequencies:
local_numerical_balance_10
-6.0      10
-5.0      12
-4.0      62
-3.0     258
-2.0     623
-1.0    1289
 0.0    1300
 1.0     261
 2.0      66
 3.0      23
 4.0      10
 NaN     246
Name: count, dtype: int64


In [176]:
all_360_events[
    "nearest_teammate_location"
] = all_360_events[
    "freeze_frame"
].apply(
    lambda x: get_nearest_teammate(x)[0]
)

all_360_events[
    "nearest_teammate_distance"
] = all_360_events[
    "freeze_frame"
].apply(
    lambda x: get_nearest_teammate(x)[1]
)

all_360_events[
    "nearby_teammates_5"
] = all_360_events[
    "freeze_frame"
].apply(
    lambda x: count_nearby_teammates(
        x,
        radius=5
    )
)

all_360_events[
    "nearby_opponents_10"
] = all_360_events[
    "freeze_frame"
].apply(
    lambda x: count_nearby_opponents(
        x,
        radius=10
    )
)

all_360_events[
    "nearby_teammates_10"
] = all_360_events[
    "freeze_frame"
].apply(
    lambda x: count_nearby_teammates(
        x,
        radius=10
    )
)

all_360_events[
    "local_numerical_balance_5"
] = (
    all_360_events["nearby_teammates_5"]
    -
    all_360_events["nearby_opponents_5"]
)

all_360_events[
    "local_numerical_balance_10"
] = (
    all_360_events["nearby_teammates_10"]
    -
    all_360_events["nearby_opponents_10"]
)

In [177]:
new_context_features = [
    "nearest_teammate_distance",
    "nearby_teammates_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_5",
    "local_numerical_balance_10"
]

print(
    all_360_events[
        new_context_features
    ].describe()
)

print(
    "\nMissing values:"
)

print(
    all_360_events[
        new_context_features
    ].isna().sum()
)

       nearest_teammate_distance  nearby_teammates_5  nearby_opponents_10  \
count               1.356304e+06        1.357625e+06         1.357625e+06   
mean                1.032351e+01        1.747427e-01         1.696955e+00   
std                 5.122511e+00        5.010433e-01         1.362906e+00   
min                 0.000000e+00        0.000000e+00         0.000000e+00   
25%                 6.678440e+00        0.000000e+00         1.000000e+00   
50%                 9.715651e+00        0.000000e+00         1.000000e+00   
75%                 1.333587e+01        0.000000e+00         2.000000e+00   
max                 7.086532e+01        9.000000e+00         1.100000e+01   

       nearby_teammates_10  local_numerical_balance_10  
count         1.357625e+06                1.357625e+06  
mean          8.969738e-01               -7.999808e-01  
std           1.171336e+00                1.261288e+00  
min           0.000000e+00               -9.000000e+00  
25%           0.00000

In [178]:
teammate_distance_missing = all_360_events[
    all_360_events["freeze_frame"].notna()
    &
    all_360_events[
        "nearest_teammate_distance"
    ].isna()
]

print(
    "Freeze frame present but "
    "nearest teammate missing:",
    len(teammate_distance_missing)
)

print(
    "\nNearby teammate counts:"
)

print(
    teammate_distance_missing[
        "nearby_teammates_10"
    ].value_counts(
        dropna=False
    )
)

Freeze frame present but nearest teammate missing: 1323

Nearby teammate counts:
nearby_teammates_10
0.0    1321
NaN       2
Name: count, dtype: int64


In [179]:
problem_teammate_cases = (
    teammate_distance_missing[
        teammate_distance_missing[
            "nearby_teammates_10"
        ].isna()
    ]
)

print(
    problem_teammate_cases[
        [
            "match_id",
            "id",
            "event_type",
            "player_name",
            "freeze_frame"
        ]
    ]
)

        match_id                                    id    event_type  \
943652   3893793  e43bc7e0-4353-4193-a104-29897bf67703         50/50   
1564695  4018356  dfe42790-bef1-4e78-9632-de2e747af5cc  Dispossessed   

            player_name                                       freeze_frame  
943652    Rion Ishikawa  [{'teammate': False, 'actor': False, 'keeper':...  
1564695  Leila Wandeler  [{'teammate': False, 'actor': False, 'keeper':...  


In [180]:
for value in problem_teammate_cases[
    "freeze_frame"
]:
    print(
        "Type:",
        type(value)
    )
    print(
        "Value:",
        value
    )
    print("---")

Type: <class 'list'>
Value: [{'teammate': False, 'actor': False, 'keeper': False, 'location': [69.04302884292304, 61.76283668021257]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [73.31509313156657, 50.31341006463371]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [75.26310998096044, 40.43722451026392]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [78.38388054233234, 52.891719465225194]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [81.3652940881248, 32.62404384746224]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [82.75306390732136, 22.32352934411339]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [84.1860379966309, 22.085267615220207]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [84.82132330885189, 42.88965840022291]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [87.5999984741211, 35.900001525878906]}, {'teammate': Fals

In [181]:
zero_teammate_distance = all_360_events[
    all_360_events[
        "nearest_teammate_distance"
    ] == 0
]

print(
    "Exact zero teammate distances:",
    len(zero_teammate_distance)
)

print(
    "Percentage of measurable distances:",
    (
        len(zero_teammate_distance)
        /
        all_360_events[
            "nearest_teammate_distance"
        ].notna().sum()
    ) * 100
)

Exact zero teammate distances: 8
Percentage of measurable distances: 0.0005898382663473675


In [183]:
from pathlib import Path

failed_file = Path(
    "all_360_context_features.pkl"
)

if failed_file.exists():
    failed_file.unlink()
    print("Removed incomplete file.")
else:
    print("No incomplete file found.")

Removed incomplete file.


In [184]:
context_feature_columns = [
    "match_id",
    "id",
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_5",
    "local_numerical_balance_10",
    "under_spatial_pressure"
]

context_360_features = (
    all_360_events[
        context_feature_columns
    ]
    .copy()
)

print(
    "Context feature shape:",
    context_360_features.shape
)

print(
    "Memory usage MB:",
    context_360_features.memory_usage(
        deep=True
    ).sum() / 1024**2
)

Context feature shape: (1582258, 11)
Memory usage MB: 476.2027339935303


In [185]:
context_feature_columns = [
    "match_id",
    "id",
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_5",
    "local_numerical_balance_10",
    "under_spatial_pressure"
]

context_360_features = (
    all_360_events[
        context_feature_columns
    ]
    .copy()
)

print(
    "Context feature shape:",
    context_360_features.shape
)

print(
    "Memory usage MB:",
    context_360_features.memory_usage(
        deep=True
    ).sum() / 1024**2
)

Context feature shape: (1582258, 11)
Memory usage MB: 476.2027339935303


In [186]:
print(
    "Duplicate match-event rows:",
    context_360_features.duplicated(
        subset=[
            "match_id",
            "id"
        ]
    ).sum()
)

print(
    "Unique matches:",
    context_360_features[
        "match_id"
    ].nunique()
)

Duplicate match-event rows: 0
Unique matches: 425


In [187]:
context_360_features.to_pickle(
    "360_context_features.pkl"
)

print("360 context features saved.")

360 context features saved.


In [188]:
["match_id", "id"]

['match_id', 'id']

In [190]:
print(
    context_360_features.duplicated(
        subset=["match_id", "id"]
    ).sum()
)

0


In [191]:
context_360_features.to_pickle(
    "360_context_features.pkl"
)

print(
    "Saved:",
    context_360_features.shape
)


Saved: (1582258, 11)


In [192]:
passes_test = (
    test_match[
        test_match["event_type"] == "Pass"
    ]
    .copy()
)

print(
    "Total passes:",
    len(passes_test)
)

print(
    "Passes with 360:",
    passes_test[
        "freeze_frame"
    ].notna().sum()
)

Total passes: 1215
Passes with 360: 1150


In [193]:
pass_model_columns = [
    "match_id",
    "id",
    "player_name",
    "pass_completed",
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_5",
    "local_numerical_balance_10",
    "under_spatial_pressure"
]

for column in pass_model_columns:
    print(
        column,
        "→",
        column in passes_test.columns
    )

match_id → True
id → True
player_name → True
pass_completed → True
nearest_opponent_distance → True
nearest_teammate_distance → True
nearby_opponents_5 → True
nearby_teammates_5 → True
nearby_opponents_10 → True
nearby_teammates_10 → True
local_numerical_balance_5 → True
local_numerical_balance_10 → True
under_spatial_pressure → True


In [194]:
print(
    "Total passes:",
    len(passes_test)
)

print(
    "Passes with 360:",
    passes_test[
        "freeze_frame"
    ].notna().sum()
)

Total passes: 1215
Passes with 360: 1150


In [195]:
possible_pass_features = [
    "pass_length",
    "pass_angle",
    "pass_angle_degrees",
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "forward_distance",
    "distance_to_goal_start",
    "distance_to_goal_end",
    "goal_distance_reduction",
    "goal_distance_reduction_pct",
    "is_forward_pass",
    "is_progressive_pass",
    "shot_assist",
    "goal_assist",
    "is_cross",
    "is_through_ball",
    "is_switch"
]

for column in possible_pass_features:
    print(
        column,
        "→",
        column in passes_test.columns
    )

pass_length → False
pass_angle → False
pass_angle_degrees → False
start_x → False
start_y → False
end_x → False
end_y → False
forward_distance → False
distance_to_goal_start → False
distance_to_goal_end → False
goal_distance_reduction → False
goal_distance_reduction_pct → False
is_forward_pass → False
is_progressive_pass → False
shot_assist → False
goal_assist → False
is_cross → False
is_through_ball → False
is_switch → False


In [196]:
import numpy as np
import pandas as pd


# -------------------------
# Pass length and angle
# -------------------------

passes_test["pass_length"] = (
    passes_test["pass"].apply(
        lambda x: x.get(
            "length",
            np.nan
        )
        if isinstance(x, dict)
        else np.nan
    )
)

passes_test["pass_angle"] = (
    passes_test["pass"].apply(
        lambda x: x.get(
            "angle",
            np.nan
        )
        if isinstance(x, dict)
        else np.nan
    )
)

passes_test[
    "pass_angle_degrees"
] = np.degrees(
    pd.to_numeric(
        passes_test["pass_angle"],
        errors="coerce"
    )
)


# -------------------------
# Pass start location
# -------------------------

passes_test["start_x"] = (
    passes_test["location"].apply(
        lambda x: x[0]
        if isinstance(x, list)
        and len(x) >= 2
        else np.nan
    )
)

passes_test["start_y"] = (
    passes_test["location"].apply(
        lambda x: x[1]
        if isinstance(x, list)
        and len(x) >= 2
        else np.nan
    )
)


# -------------------------
# Pass end location
# -------------------------

def get_pass_end_coordinate(
    pass_data,
    index
):

    if not isinstance(
        pass_data,
        dict
    ):
        return np.nan

    end_location = pass_data.get(
        "end_location"
    )

    if (
        isinstance(
            end_location,
            list
        )
        and len(end_location) >= 2
    ):
        return end_location[index]

    return np.nan


passes_test["end_x"] = (
    passes_test["pass"].apply(
        lambda x:
        get_pass_end_coordinate(
            x,
            0
        )
    )
)

passes_test["end_y"] = (
    passes_test["pass"].apply(
        lambda x:
        get_pass_end_coordinate(
            x,
            1
        )
    )
)


# -------------------------
# Forward distance
# -------------------------

passes_test[
    "forward_distance"
] = (
    passes_test["end_x"]
    -
    passes_test["start_x"]
)

passes_test[
    "is_forward_pass"
] = (
    passes_test[
        "forward_distance"
    ] > 0
)


# -------------------------
# Distance to opponent goal
# StatsBomb pitch:
# x = 0 to 120
# y = 0 to 80
# opponent goal centre = (120, 40)
# -------------------------

passes_test[
    "distance_to_goal_start"
] = np.sqrt(
    (
        120
        - passes_test["start_x"]
    ) ** 2
    +
    (
        40
        - passes_test["start_y"]
    ) ** 2
)

passes_test[
    "distance_to_goal_end"
] = np.sqrt(
    (
        120
        - passes_test["end_x"]
    ) ** 2
    +
    (
        40
        - passes_test["end_y"]
    ) ** 2
)

passes_test[
    "goal_distance_reduction"
] = (
    passes_test[
        "distance_to_goal_start"
    ]
    -
    passes_test[
        "distance_to_goal_end"
    ]
)

passes_test[
    "goal_distance_reduction_pct"
] = np.where(
    passes_test[
        "distance_to_goal_start"
    ] > 0,

    (
        passes_test[
            "goal_distance_reduction"
        ]
        /
        passes_test[
            "distance_to_goal_start"
        ]
    ) * 100,

    np.nan
)


# -------------------------
# Progressive pass
# Working project definition:
# >= 25% reduction in distance
# to opponent goal
# -------------------------

passes_test[
    "is_progressive_pass"
] = (
    passes_test[
        "goal_distance_reduction_pct"
    ] >= 25
)


# -------------------------
# Pass type/context flags
# -------------------------

passes_test[
    "shot_assist"
] = passes_test["pass"].apply(
    lambda x:
    x.get(
        "shot_assist",
        False
    )
    if isinstance(x, dict)
    else False
)

passes_test[
    "goal_assist"
] = passes_test["pass"].apply(
    lambda x:
    x.get(
        "goal_assist",
        False
    )
    if isinstance(x, dict)
    else False
)

passes_test[
    "is_cross"
] = passes_test["pass"].apply(
    lambda x:
    x.get(
        "cross",
        False
    )
    if isinstance(x, dict)
    else False
)

passes_test[
    "is_through_ball"
] = passes_test["pass"].apply(
    lambda x:
    x.get(
        "through_ball",
        False
    )
    if isinstance(x, dict)
    else False
)

passes_test[
    "is_switch"
] = passes_test["pass"].apply(
    lambda x:
    x.get(
        "switch",
        False
    )
    if isinstance(x, dict)
    else False
)

In [197]:
check_pass_features = [
    "pass_completed",
    "pass_length",
    "pass_angle_degrees",
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "forward_distance",
    "goal_distance_reduction_pct",
    "is_forward_pass",
    "is_progressive_pass",
    "shot_assist",
    "goal_assist",
    "is_cross",
    "is_through_ball",
    "is_switch"
]

print(
    passes_test[
        check_pass_features
    ].head(10)
)

   pass_completed  pass_length  pass_angle_degrees  start_x  start_y  end_x  \
4            True    15.597436          139.159646     61.0     40.1   49.2   
7            True    17.520845         -162.392362     50.1     51.4   33.4   
10           True    27.011848          -88.302846     34.3     45.3   35.1   
13           True    33.664370          103.747290     34.5     22.8   26.5   
16           True    55.965080           -7.391684     34.1     63.3   89.6   
21           True    23.493190          -56.986354     12.3     34.6   25.1   
24           True    33.065086           98.522201     27.3     17.2   22.4   
27           True    15.897799            9.047571     28.2     53.6   43.9   
30           True    35.203552          -90.813805     43.9     56.1   43.4   
33           True    22.249720          110.797163     51.3     19.5   43.4   

    end_y  forward_distance  goal_distance_reduction_pct  is_forward_pass  \
4    50.3             -11.8                   -21.263

In [198]:
print(
    "Total passes:",
    len(passes_test)
)

print(
    "Completed:",
    (passes_test["pass_completed"] == True).sum()
)

print(
    "Incomplete:",
    (passes_test["pass_completed"] == False).sum()
)

print(
    "Progressive passes:",
    passes_test["is_progressive_pass"].sum()
)

print(
    "Missing pass length:",
    passes_test["pass_length"].isna().sum()
)

print(
    "Missing end location:",
    passes_test["end_x"].isna().sum()
)

Total passes: 1215
Completed: 1083
Incomplete: 132
Progressive passes: 134
Missing pass length: 0
Missing end location: 0


In [199]:
pass_model_test = passes_test[
    [
        # Identifiers
        "match_id",
        "id",
        "player_name",

        # Target
        "pass_completed",

        # Pass difficulty
        "pass_length",
        "pass_angle_degrees",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "forward_distance",
        "goal_distance_reduction_pct",
        "is_forward_pass",
        "is_progressive_pass",
        "is_cross",
        "is_through_ball",
        "is_switch",

        # Immediate spatial context
        "nearest_opponent_distance",
        "nearest_teammate_distance",
        "under_spatial_pressure",

        # 5-unit context
        "nearby_opponents_5",
        "nearby_teammates_5",
        "local_numerical_balance_5",

        # 10-unit context
        "nearby_opponents_10",
        "nearby_teammates_10",
        "local_numerical_balance_10"
    ]
].copy()

print(
    "Model table shape:",
    pass_model_test.shape
)

print(
    "\nMissing values:"
)

print(
    pass_model_test.isna().sum()
)

Model table shape: (1215, 26)

Missing values:
match_id                        0
id                              0
player_name                     0
pass_completed                  0
pass_length                     0
pass_angle_degrees              0
start_x                         0
start_y                         0
end_x                           0
end_y                           0
forward_distance                0
goal_distance_reduction_pct     0
is_forward_pass                 0
is_progressive_pass             0
is_cross                        0
is_through_ball                 0
is_switch                       0
nearest_opponent_distance      65
nearest_teammate_distance      65
under_spatial_pressure         65
nearby_opponents_5             65
nearby_teammates_5             65
local_numerical_balance_5      65
nearby_opponents_10            65
nearby_teammates_10            65
local_numerical_balance_10     65
dtype: int64


In [200]:
spatial_columns = [
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "under_spatial_pressure",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "local_numerical_balance_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_10"
]

missing_spatial_count = (
    pass_model_test[
        spatial_columns
    ]
    .isna()
    .any(axis=1)
    .sum()
)

complete_spatial_count = (
    pass_model_test[
        spatial_columns
    ]
    .notna()
    .all(axis=1)
    .sum()
)

print(
    "Passes with any spatial feature missing:",
    missing_spatial_count
)

print(
    "Passes with complete spatial context:",
    complete_spatial_count
)

print(
    "Spatial coverage:",
    (
        complete_spatial_count
        / len(pass_model_test)
    ) * 100
)

Passes with any spatial feature missing: 65
Passes with complete spatial context: 1150
Spatial coverage: 94.65020576131687


In [201]:
all_360_passes = (
    all_360_events[
        all_360_events["event_type"] == "Pass"
    ]
    .copy()
)

print(
    "Total 360-match passes:",
    len(all_360_passes)
)

print(
    "Unique matches:",
    all_360_passes["match_id"].nunique()
)

print(
    "Memory usage MB:",
    all_360_passes.memory_usage(
        deep=True
    ).sum() / 1024**2
)

Total 360-match passes: 447122
Unique matches: 425
Memory usage MB: 1556.3402242660522


In [202]:
pass_working = all_360_passes[
    [
        "match_id",
        "id",
        "player",
        "player_name",
        "location",
        "pass",
        "pass_completed"
    ]
].copy()

print(
    "Compact pass table shape:",
    pass_working.shape
)

print(
    "Memory usage MB:",
    pass_working.memory_usage(
        deep=True
    ).sum() / 1024**2
)

Compact pass table shape: (447122, 7)
Memory usage MB: 368.82760524749756


In [203]:
pass_working["player_id"] = (
    pass_working["player"].apply(
        lambda x:
        x.get("id")
        if isinstance(x, dict)
        else pd.NA
    )
)

pass_working["player_id"] = (
    pd.to_numeric(
        pass_working["player_id"],
        errors="coerce"
    ).astype("Int64")
)

print(
    "Missing player IDs:",
    pass_working["player_id"].isna().sum()
)

print(
    "Unique players:",
    pass_working["player_id"].nunique()
)

Missing player IDs: 0
Unique players: 3263


In [204]:
def prepare_pass_features(pass_df):

    df = pass_df.copy()

    # Pass length
    df["pass_length"] = df["pass"].apply(
        lambda x: x.get("length", np.nan)
        if isinstance(x, dict)
        else np.nan
    )

    # Pass angle
    df["pass_angle"] = df["pass"].apply(
        lambda x: x.get("angle", np.nan)
        if isinstance(x, dict)
        else np.nan
    )

    df["pass_angle_degrees"] = np.degrees(
        pd.to_numeric(
            df["pass_angle"],
            errors="coerce"
        )
    )

    # Start location
    df["start_x"] = df["location"].apply(
        lambda x: x[0]
        if isinstance(x, list) and len(x) >= 2
        else np.nan
    )

    df["start_y"] = df["location"].apply(
        lambda x: x[1]
        if isinstance(x, list) and len(x) >= 2
        else np.nan
    )

    # End location
    def get_end_coordinate(pass_data, index):

        if not isinstance(pass_data, dict):
            return np.nan

        end_location = pass_data.get(
            "end_location"
        )

        if (
            isinstance(end_location, list)
            and len(end_location) >= 2
        ):
            return end_location[index]

        return np.nan

    df["end_x"] = df["pass"].apply(
        lambda x: get_end_coordinate(x, 0)
    )

    df["end_y"] = df["pass"].apply(
        lambda x: get_end_coordinate(x, 1)
    )

    # Forward distance
    df["forward_distance"] = (
        df["end_x"]
        - df["start_x"]
    )

    df["is_forward_pass"] = (
        df["forward_distance"] > 0
    )

    # Distance to opponent goal
    df["distance_to_goal_start"] = np.sqrt(
        (120 - df["start_x"]) ** 2
        +
        (40 - df["start_y"]) ** 2
    )

    df["distance_to_goal_end"] = np.sqrt(
        (120 - df["end_x"]) ** 2
        +
        (40 - df["end_y"]) ** 2
    )

    df["goal_distance_reduction"] = (
        df["distance_to_goal_start"]
        -
        df["distance_to_goal_end"]
    )

    df["goal_distance_reduction_pct"] = np.where(
        df["distance_to_goal_start"] > 0,
        (
            df["goal_distance_reduction"]
            /
            df["distance_to_goal_start"]
        ) * 100,
        np.nan
    )

    # Progressive pass
    df["is_progressive_pass"] = (
        df["goal_distance_reduction_pct"]
        >= 25
    )

    # Pass-context flags
    df["is_cross"] = df["pass"].apply(
        lambda x: x.get("cross", False)
        if isinstance(x, dict)
        else False
    )

    df["is_through_ball"] = df["pass"].apply(
        lambda x: x.get("through_ball", False)
        if isinstance(x, dict)
        else False
    )

    df["is_switch"] = df["pass"].apply(
        lambda x: x.get("switch", False)
        if isinstance(x, dict)
        else False
    )

    return df

In [205]:
pass_features_full = prepare_pass_features(
    pass_working
)

print(
    "Pass feature shape:",
    pass_features_full.shape
)

Pass feature shape: (447122, 25)


In [206]:
print(
    "Missing pass length:",
    pass_features_full[
        "pass_length"
    ].isna().sum()
)

print(
    "Missing end location:",
    pass_features_full[
        "end_x"
    ].isna().sum()
)

print(
    "Progressive passes:",
    pass_features_full[
        "is_progressive_pass"
    ].sum()
)

print(
    "Completed passes:",
    (
        pass_features_full[
            "pass_completed"
        ] == True
    ).sum()
)

print(
    "Incomplete passes:",
    (
        pass_features_full[
            "pass_completed"
        ] == False
    ).sum()
)

Missing pass length: 0
Missing end location: 0
Progressive passes: 76921
Completed passes: 367768
Incomplete passes: 79354


In [207]:
context_360_features = pd.read_pickle(
    "360_context_features.pkl"
)

pass_model_full = pass_features_full.merge(
    context_360_features,
    on=[
        "match_id",
        "id"
    ],
    how="left",
    validate="one_to_one"
)

print(
    "Merged shape:",
    pass_model_full.shape
)

print(
    "Duplicate match-event rows:",
    pass_model_full.duplicated(
        subset=[
            "match_id",
            "id"
        ]
    ).sum()
)

Merged shape: (447122, 34)
Duplicate match-event rows: 0


In [208]:
spatial_columns = [
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "under_spatial_pressure",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "local_numerical_balance_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_10"
]

print(
    "\nMissing spatial values:"
)

print(
    pass_model_full[
        spatial_columns
    ].isna().sum()
)

print(
    "\nPasses with complete spatial context:",
    pass_model_full[
        spatial_columns
    ]
    .notna()
    .all(axis=1)
    .sum()
)

print(
    "Passes with any spatial feature missing:",
    pass_model_full[
        spatial_columns
    ]
    .isna()
    .any(axis=1)
    .sum()
)


Missing spatial values:
nearest_opponent_distance     74052
nearest_teammate_distance     73795
under_spatial_pressure        74052
nearby_opponents_5            73482
nearby_teammates_5            73482
local_numerical_balance_5     73482
nearby_opponents_10           73482
nearby_teammates_10           73482
local_numerical_balance_10    73482
dtype: int64

Passes with complete spatial context: 372982
Passes with any spatial feature missing: 74140


In [209]:
pass_model_full[
    "has_basic_360_context"
] = (
    pass_model_full[
        "nearby_opponents_5"
    ].notna()
)

pass_model_full[
    "has_complete_360_context"
] = (
    pass_model_full[
        spatial_columns
    ]
    .notna()
    .all(axis=1)
)

print(
    "Total passes:",
    len(pass_model_full)
)

print(
    "Basic 360 context available:",
    pass_model_full[
        "has_basic_360_context"
    ].sum()
)

print(
    "Complete 360 context available:",
    pass_model_full[
        "has_complete_360_context"
    ].sum()
)

print(
    "Basic context but incomplete distances:",
    (
        pass_model_full[
            "has_basic_360_context"
        ]
        &
        ~pass_model_full[
            "has_complete_360_context"
        ]
    ).sum()
)

print(
    "No usable basic 360 context:",
    (
        ~pass_model_full[
            "has_basic_360_context"
        ]
    ).sum()
)

Total passes: 447122
Basic 360 context available: 373640
Complete 360 context available: 372982
Basic context but incomplete distances: 658
No usable basic 360 context: 73482


In [210]:
pass_model_full["match_id"] = pd.to_numeric(
    pass_model_full["match_id"],
    errors="coerce"
).astype("Int64")

pass_model_full["player_id"] = pd.to_numeric(
    pass_model_full["player_id"],
    errors="coerce"
).astype("Int64")

print(
    "Missing match IDs:",
    pass_model_full["match_id"].isna().sum()
)

print(
    "Missing player IDs:",
    pass_model_full["player_id"].isna().sum()
)

Missing match IDs: 0
Missing player IDs: 0


In [211]:
pass_model_full = pass_model_full.merge(
    match_metadata[
        [
            "match_id",
            "match_date"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

print(
    "Pass rows:",
    len(pass_model_full)
)

print(
    "Passes missing match date:",
    pass_model_full[
        "match_date"
    ].isna().sum()
)

print(
    "Matches missing date:",
    pass_model_full.loc[
        pass_model_full[
            "match_date"
        ].isna(),
        "match_id"
    ].nunique()
)

Pass rows: 447122
Passes missing match date: 0
Matches missing date: 0


In [212]:
historical_columns = [
    col
    for col in historical_model_features.columns
    if (
        "previous_reliable" in col
        or "last3_reliable" in col
        or "last5_reliable" in col
        or col == "previous_matches_available"
    )
]

print("Historical columns available:")
for col in historical_columns:
    print(col)

print(
    "\nNumber of selected columns:",
    len(historical_columns)
)

Historical columns available:
previous_matches_available
previous_reliable_match_id
previous_reliable_pass_attempts_per90
previous_reliable_progressive_passes_per90
previous_reliable_carries_per90
previous_reliable_shots_per90
previous_reliable_miscontrols_per90
previous_reliable_shot_assists_per90

Number of selected columns: 8


In [213]:
pass_completion_history_cols = [
    col
    for col in historical_model_features.columns
    if "pass_completion" in col.lower()
]

print("Pass-completion history columns:")

for col in pass_completion_history_cols:
    print(col)

Pass-completion history columns:
pass_completion_rate
prev_pass_completion_rate
rolling_3_pass_completion
rolling_5_pass_completion


In [214]:
rolling_history_cols = [
    col
    for col in historical_model_features.columns
    if (
        col.startswith("prev_")
        or col.startswith("rolling_3_")
        or col.startswith("rolling_5_")
    )
]

print("\nPrevious / rolling historical columns:")

for col in rolling_history_cols:
    print(col)


Previous / rolling historical columns:
prev_pass_completion_rate
rolling_3_pass_completion
rolling_5_pass_completion
prev_pass_attempts
rolling_3_pass_attempts
rolling_5_pass_attempts
prev_progressive_passes
rolling_3_progressive_passes
rolling_5_progressive_passes
prev_carries
rolling_3_carries
rolling_5_carries
prev_shots
rolling_3_shots
rolling_5_shots
prev_miscontrols
rolling_3_miscontrols
rolling_5_miscontrols
prev_shot_assists
rolling_3_shot_assists
rolling_5_shot_assists


In [215]:
historical_pass_columns = [
    "match_id",
    "player_id",

    # Historical passing accuracy
    "prev_pass_completion_rate",
    "rolling_3_pass_completion",
    "rolling_5_pass_completion",

    # Amount of previous history available
    "previous_matches_available",

    # Reliable activity baselines
    "previous_reliable_pass_attempts_per90",
    "previous_reliable_progressive_passes_per90",
    "previous_reliable_carries_per90",
    "previous_reliable_shots_per90",
    "previous_reliable_miscontrols_per90",
    "previous_reliable_shot_assists_per90"
]

historical_pass_features = (
    historical_model_features[
        historical_pass_columns
    ]
    .copy()
)

print(
    "Historical feature table shape:",
    historical_pass_features.shape
)

print(
    "Duplicate match-player rows:",
    historical_pass_features.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

Historical feature table shape: (113470, 12)
Duplicate match-player rows: 0


In [216]:
print(
    historical_pass_features[
        ["match_id", "player_id"]
    ].dtypes
)

print(
    pass_model_full[
        ["match_id", "player_id"]
    ].dtypes
)

match_id     int64
player_id    Int64
dtype: object
match_id     Int64
player_id    Int64
dtype: object


In [217]:
historical_pass_features["match_id"] = pd.to_numeric(
    historical_pass_features["match_id"],
    errors="coerce"
).astype("Int64")

historical_pass_features["player_id"] = pd.to_numeric(
    historical_pass_features["player_id"],
    errors="coerce"
).astype("Int64")

pass_model_full["match_id"] = pd.to_numeric(
    pass_model_full["match_id"],
    errors="coerce"
).astype("Int64")

pass_model_full["player_id"] = pd.to_numeric(
    pass_model_full["player_id"],
    errors="coerce"
).astype("Int64")

print(
    historical_pass_features[
        ["match_id", "player_id"]
    ].dtypes
)

print(
    pass_model_full[
        ["match_id", "player_id"]
    ].dtypes
)

match_id     Int64
player_id    Int64
dtype: object
match_id     Int64
player_id    Int64
dtype: object


In [218]:
duplicate_history = (
    historical_pass_features
    .duplicated(
        subset=["match_id", "player_id"]
    )
    .sum()
)

print(
    "Duplicate historical match-player rows:",
    duplicate_history
)

Duplicate historical match-player rows: 0


In [219]:
rows_before = len(pass_model_full)

pass_model_full = pass_model_full.merge(
    historical_pass_features,
    on=["match_id", "player_id"],
    how="left",
    validate="many_to_one"
)

rows_after = len(pass_model_full)

print("Rows before merge:", rows_before)
print("Rows after merge:", rows_after)

print(
    "Rows added/lost:",
    rows_after - rows_before
)

Rows before merge: 447122
Rows after merge: 447122
Rows added/lost: 0


In [220]:
print(
    "\nMissing historical features:"
)

history_check_columns = [
    "prev_pass_completion_rate",
    "rolling_3_pass_completion",
    "rolling_5_pass_completion",
    "previous_matches_available",
    "previous_reliable_pass_attempts_per90"
]

print(
    pass_model_full[
        history_check_columns
    ].isna().sum()
)

print(
    "\nPasses with previous reliable history:",
    pass_model_full[
        "previous_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Passes without previous reliable history:",
    pass_model_full[
        "previous_reliable_pass_attempts_per90"
    ].isna().sum()
)


Missing historical features:
prev_pass_completion_rate                51077
rolling_3_pass_completion                50053
rolling_5_pass_completion                50053
previous_matches_available                   0
previous_reliable_pass_attempts_per90    55459
dtype: int64

Passes with previous reliable history: 391663
Passes without previous reliable history: 55459


In [221]:
pass_volume_history = historical_model_features[
    [
        "match_id",
        "player_id",
        "prev_pass_attempts",
        "rolling_3_pass_attempts",
        "rolling_5_pass_attempts"
    ]
].copy()

pass_volume_history["match_id"] = pd.to_numeric(
    pass_volume_history["match_id"],
    errors="coerce"
).astype("Int64")

pass_volume_history["player_id"] = pd.to_numeric(
    pass_volume_history["player_id"],
    errors="coerce"
).astype("Int64")

print(
    "Duplicate match-player rows:",
    pass_volume_history.duplicated(
        ["match_id", "player_id"]
    ).sum()
)

Duplicate match-player rows: 0


In [222]:
pass_model_full = pass_model_full.merge(
    pass_volume_history,
    on=["match_id", "player_id"],
    how="left",
    validate="many_to_one"
)

print(
    pass_model_full[
        [
            "prev_pass_completion_rate",
            "prev_pass_attempts",
            "rolling_3_pass_completion",
            "rolling_3_pass_attempts",
            "rolling_5_pass_completion",
            "rolling_5_pass_attempts"
        ]
    ].isna().sum()
)

prev_pass_completion_rate    51077
prev_pass_attempts           49726
rolling_3_pass_completion    50053
rolling_3_pass_attempts      49726
rolling_5_pass_completion    50053
rolling_5_pass_attempts      49726
dtype: int64


In [223]:
event_features_strict = [
    "pass_length",
    "pass_angle_degrees",
    "start_x",
    "start_y",
    "is_cross",
    "is_through_ball",
    "is_switch"
]

spatial_features = [
    "nearest_opponent_distance",
    "nearest_teammate_distance",
    "under_spatial_pressure",
    "nearby_opponents_5",
    "nearby_teammates_5",
    "local_numerical_balance_5",
    "nearby_opponents_10",
    "nearby_teammates_10",
    "local_numerical_balance_10"
]

historical_features = [
    "prev_pass_completion_rate",
    "rolling_3_pass_completion",
    "rolling_5_pass_completion",
    "prev_pass_attempts",
    "rolling_3_pass_attempts",
    "rolling_5_pass_attempts",
    "previous_matches_available",
    "previous_reliable_pass_attempts_per90",
    "previous_reliable_progressive_passes_per90",
    "previous_reliable_carries_per90",
    "previous_reliable_shots_per90",
    "previous_reliable_miscontrols_per90",
    "previous_reliable_shot_assists_per90"
]

endpoint_features = [
    "end_x",
    "end_y",
    "forward_distance",
    "goal_distance_reduction_pct",
    "is_forward_pass",
    "is_progressive_pass"
]

print("Strict event features:", len(event_features_strict))
print("360 spatial features:", len(spatial_features))
print("Historical features:", len(historical_features))
print("Endpoint-derived features:", len(endpoint_features))

print(
    "\nTotal core predictors:",
    len(
        event_features_strict
        + spatial_features
        + historical_features
    )
)

Strict event features: 7
360 spatial features: 9
Historical features: 13
Endpoint-derived features: 6

Total core predictors: 29


In [225]:
identifier_columns = [
    "match_id",
    "id",
    "player_id",
    "player_name",
    "match_date"
]

target_column = [
    "pass_completed"
]

final_model_columns = (
    identifier_columns
    + target_column
    + event_features_strict
    + spatial_features
    + historical_features
    + endpoint_features
)

final_pass_model_data = (
    pass_model_full[
        final_model_columns
    ]
    .copy()
)

print(
    "Final modelling dataset shape:",
    final_pass_model_data.shape
)

print(
    "Duplicate pass events:",
    final_pass_model_data.duplicated(
        subset=["match_id", "id"]
    ).sum()
)

print(
    "Missing target:",
    final_pass_model_data[
        "pass_completed"
    ].isna().sum()
)

print(
    "Unique matches:",
    final_pass_model_data[
        "match_id"
    ].nunique()
)

print(
    "Unique players:",
    final_pass_model_data[
        "player_id"
    ].nunique()
)

Final modelling dataset shape: (447122, 41)
Duplicate pass events: 0
Missing target: 0
Unique matches: 425
Unique players: 3263


In [226]:
print(
    final_pass_model_data[
        "pass_completed"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nCompletion rate:"
)

print(
    final_pass_model_data[
        "pass_completed"
    ].mean() * 100
)


pass_completed
True     367768
False     79354
Name: count, dtype: int64

Completion rate:
82.25227119220257


In [227]:
final_pass_model_data.to_pickle(
    "final_pass_modelling_dataset.pkl"
)

print(
    "Saved final_pass_modelling_dataset.pkl"
)

print(
    "Memory usage MB:",
    final_pass_model_data.memory_usage(
        deep=True
    ).sum() / 1024**2
)

Saved final_pass_modelling_dataset.pkl
Memory usage MB: 245.00162601470947


## Historical Feature Regeneration Using Corrected Player Exposure

During downstream validation of the player forecasting pipeline, edge cases were identified in the original player-minutes calculation. A revised interval-based exposure method was therefore developed and validated in Notebook 02. The historical minutes-dependent features are regenerated below using the corrected exposure dataset.

In [1]:
import pandas as pd

corrected_player_minutes = pd.read_pickle(
    "all_player_exposure_corrected.pkl"
)

print("Shape:", corrected_player_minutes.shape)
print("Matches:", corrected_player_minutes["match_id"].nunique())
print("Players:", corrected_player_minutes["player_id"].nunique())

display(corrected_player_minutes.head())

Shape: (121214, 13)
Matches: 4235
Players: 10006


,match_id,player_id,player_name,team_name,minutes_played,first_half_minutes,second_half_minutes,extra_time_minutes,temporary_off_pitch_gaps,match_duration_minutes,first_half_duration,second_half_duration,playing_intervals_seconds
0,15946,20055,Marc-André ter Stegen,Barcelona,92.616667,45.1,47.516667,0.0,0,92.616667,45.1,47.516667,"[(0, 5557.0)]"
1,15946,6374,Nélson Cabral Semedo,Barcelona,45.100000,45.1,0.000000,0.0,0,92.616667,45.1,47.516667,"[(0, 2706.0)]"
2,15946,5213,Gerard Piqué Bernabéu,Barcelona,92.616667,45.1,47.516667,0.0,0,92.616667,45.1,47.516667,"[(0, 5557.0)]"
3,15946,5492,Samuel Yves Umtiti,Barcelona,92.616667,45.1,47.516667,0.0,0,92.616667,45.1,47.516667,"[(0, 5557.0)]"
4,15946,5211,Jordi Alba Ramos,Barcelona,92.616667,45.1,47.516667,0.0,0,92.616667,45.1,47.516667,"[(0, 5557.0)]"


In [3]:
historical_clean = pd.read_pickle(
    "historical_player_match_master.pkl"
)

print("Shape:", historical_clean.shape)
print("Matches:", historical_clean["match_id"].nunique())
print("Players:", historical_clean["player_id"].nunique())

print("\nColumns:")
print(historical_clean.columns.tolist())

Shape: (113470, 99)
Matches: 3961
Players: 9884

Columns:
['match_id', 'player_id', 'player_name', 'total_events', 'pass_attempts', 'passes_completed', 'forward_passes', 'progressive_passes', 'avg_pass_length', 'avg_pass_angle_degrees', 'crosses', 'through_balls', 'switches', 'shot_assists', 'goal_assists', 'shots', 'carries', 'miscontrols', 'pass_completion_rate', 'match_date', 'home_team_name', 'away_team_name', 'prev_pass_completion_rate', 'rolling_3_pass_completion', 'rolling_5_pass_completion', 'prev_pass_attempts', 'rolling_3_pass_attempts', 'rolling_5_pass_attempts', 'prev_progressive_passes', 'rolling_3_progressive_passes', 'rolling_5_progressive_passes', 'prev_carries', 'rolling_3_carries', 'rolling_5_carries', 'prev_shots', 'rolling_3_shots', 'rolling_5_shots', 'prev_miscontrols', 'rolling_3_miscontrols', 'rolling_5_miscontrols', 'prev_shot_assists', 'rolling_3_shot_assists', 'rolling_5_shot_assists', 'previous_matches_available', 'minutes_played', 'position_changes', 'tempor

In [ ]:


historical_base = historical_clean.loc[
    :,
    :"previous_matches_available"
].copy()

print("Original shape:", historical_clean.shape)
print("Clean historical base shape:", historical_base.shape)

print("\nLast 10 columns:")
print(historical_base.columns[-10:].tolist())

print(
    "\nDuplicate match-player rows:",
    historical_base.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

print(
    "Missing player IDs:",
    historical_base["player_id"].isna().sum()
)

print(
    "Missing match dates:",
    historical_base["match_date"].isna().sum()
)

Original shape: (113470, 99)
Clean historical base shape: (113470, 44)

Last 10 columns:
['prev_shots', 'rolling_3_shots', 'rolling_5_shots', 'prev_miscontrols', 'rolling_3_miscontrols', 'rolling_5_miscontrols', 'prev_shot_assists', 'rolling_3_shot_assists', 'rolling_5_shot_assists', 'previous_matches_available']

Duplicate match-player rows: 0
Missing player IDs: 0
Missing match dates: 0


In [ ]:
# ============================================================
# THIS MERGES CORRECTED PLAYER EXPOSURE INTO HISTORICAL BASE
# ============================================================

historical_base["match_id"] = (
    historical_base["match_id"]
    .astype(str)
)

corrected_player_minutes["match_id"] = (
    corrected_player_minutes["match_id"]
    .astype(str)
)

historical_base["player_id"] = pd.to_numeric(
    historical_base["player_id"],
    errors="coerce"
).astype("Int64")

corrected_player_minutes["player_id"] = pd.to_numeric(
    corrected_player_minutes["player_id"],
    errors="coerce"
).astype("Int64")


historical_with_corrected_minutes = historical_base.merge(
    corrected_player_minutes[
        [
            "match_id",
            "player_id",
            "minutes_played",
            "first_half_minutes",
            "second_half_minutes",
            "extra_time_minutes",
            "temporary_off_pitch_gaps",
            "match_duration_minutes"
        ]
    ],
    on=[
        "match_id",
        "player_id"
    ],
    how="left",
    validate="one_to_one"
)


print(
    "Shape:",
    historical_with_corrected_minutes.shape
)

print(
    "Missing corrected minutes:",
    historical_with_corrected_minutes[
        "minutes_played"
    ].isna().sum()
)

print(
    "Duplicate match-player rows:",
    historical_with_corrected_minutes.duplicated(
        subset=[
            "match_id",
            "player_id"
        ]
    ).sum()
)

print(
    "\nCorrected minutes summary:"
)

print(
    historical_with_corrected_minutes[
        "minutes_played"
    ].describe()
)

Shape: (113470, 50)
Missing corrected minutes: 37
Duplicate match-player rows: 0

Corrected minutes summary:
count    113433.000000
mean         74.198368
std          30.541909
min           0.266667
25%          52.300000
50%          93.033333
75%          96.000000
max         141.366667
Name: minutes_played, dtype: float64


In [ ]:
# ============================================================
# THIS INSPECTS THE 37 HISTORICAL ROWS WITH NO CORRECTED EXPOSURE
# ============================================================

missing_exposure = historical_with_corrected_minutes[
    historical_with_corrected_minutes["minutes_played"].isna()
].copy()

print("Missing exposure rows:", len(missing_exposure))
print("Matches involved:", missing_exposure["match_id"].nunique())
print("Players involved:", missing_exposure["player_id"].nunique())

display(
    missing_exposure[
        [
            "match_id",
            "player_id",
            "player_name",
            "match_date",
            "total_events",
            "pass_attempts",
            "shots",
            "carries",
            "miscontrols"
        ]
    ]
    .sort_values(
        ["match_id", "player_name"]
    )
)

Missing exposure rows: 37
Matches involved: 33
Players involved: 36


,match_id,player_id,player_name,match_date,total_events,pass_attempts,shots,carries,miscontrols
73088,2275049,24747,Millie Laura Farrow,2019-11-24,1,0,0,0,0
78946,266491,26686,Jordi Codina Rodríguez,2014-05-03,1,0,0,0,0
35723,303615,6758,Víctor Sánchez Mata,2020-07-08,1,0,0,0,0
23267,3773497,5213,Gerard Piqué Bernabéu,2021-04-10,1,0,0,0,0
80264,3825657,27348,Ricky van Wolfswinkel,2015-11-07,1,0,0,0,0
76752,3825670,26023,Juan Francisco García García,2015-11-27,1,0,0,0,0
103338,3825742,158769,Ivan Kelava,2016-01-23,1,0,0,0,0
41363,3825743,7105,Fabián Ariel Orellana Valenzuela,2016-01-23,1,0,0,0,0
75686,3825799,25859,Sérgio Paulo Barbosa Valente,2016-03-02,1,0,0,0,0
80350,3825804,27457,Manuel Fernández Muñíz,2016-03-05,1,0,0,0,0


In [7]:
old_exposure_check = historical_clean[
    [
        "match_id",
        "player_id",
        "minutes_played"
    ]
].copy()

old_exposure_check["match_id"] = (
    old_exposure_check["match_id"].astype(str)
)

old_exposure_check["player_id"] = pd.to_numeric(
    old_exposure_check["player_id"],
    errors="coerce"
).astype("Int64")

missing_exposure_check = missing_exposure.merge(
    old_exposure_check.rename(
        columns={
            "minutes_played": "old_minutes_played"
        }
    ),
    on=["match_id", "player_id"],
    how="left",
    validate="one_to_one"
)

display(
    missing_exposure_check[
        [
            "match_id",
            "player_id",
            "player_name",
            "total_events",
            "old_minutes_played"
        ]
    ].sort_values(
        "old_minutes_played",
        ascending=False
    )
)

,match_id,player_id,player_name,total_events,old_minutes_played
0,3920413,3417,Saidy Janko,1,0.0
1,3773497,5213,Gerard Piqué Bernabéu,1,0.0
2,3857259,6319,Luka Jović,1,0.0
3,3825869,6672,Jorge Andújar Moreno,1,0.0
4,303615,6758,Víctor Sánchez Mata,1,0.0
5,3930184,7044,Patrik Schick,1,0.0
6,3825743,7105,Fabián Ariel Orellana Valenzuela,1,0.0
7,3912527,10371,Elisa Bartoli,1,0.0
8,3900528,10448,Adam Ounas,1,0.0
9,3825885,10612,Yoel Rodríguez Oterino,1,0.0


In [8]:
import json
from pathlib import Path
import pandas as pd

events_dir = Path("open-data-master") / "data" / "events"

missing_event_details = []

for _, row in missing_exposure.iterrows():

    match_id = str(row["match_id"])
    player_id = int(row["player_id"])

    file_path = events_dir / f"{match_id}.json"

    with open(file_path, "r", encoding="utf-8") as f:
        events = json.load(f)

    for event in events:

        player = event.get("player")

        if (
            isinstance(player, dict)
            and player.get("id") == player_id
        ):

            missing_event_details.append(
                {
                    "match_id": match_id,
                    "player_id": player_id,
                    "player_name": player.get("name"),
                    "period": event.get("period"),
                    "minute": event.get("minute"),
                    "second": event.get("second"),
                    "event_type": (
                        event.get("type", {}).get("name")
                    ),
                    "position": (
                        event.get("position", {}).get("name")
                        if isinstance(event.get("position"), dict)
                        else None
                    )
                }
            )

missing_event_details = pd.DataFrame(
    missing_event_details
)

print("Events found:", len(missing_event_details))

print("\nEvent types:")
print(
    missing_event_details[
        "event_type"
    ].value_counts(dropna=False)
)

display(
    missing_event_details.sort_values(
        ["event_type", "match_id", "player_name"]
    )
)

Events found: 37

Event types:
event_type
Bad Behaviour    37
Name: count, dtype: int64


,match_id,player_id,player_name,period,minute,second,event_type,position
14,2275049,24747,Millie Laura Farrow,2,94,22,Bad Behaviour,Substitute
20,266491,26686,Jordi Codina Rodríguez,2,93,37,Bad Behaviour,Substitute
4,303615,6758,Víctor Sánchez Mata,1,45,18,Bad Behaviour,Substitute
1,3773497,5213,Gerard Piqué Bernabéu,2,94,5,Bad Behaviour,Substitute
21,3825657,27348,Ricky van Wolfswinkel,2,91,2,Bad Behaviour,Substitute
18,3825670,26023,Juan Francisco García García,2,80,16,Bad Behaviour,Substitute
35,3825742,158769,Ivan Kelava,2,76,47,Bad Behaviour,Substitute
6,3825743,7105,Fabián Ariel Orellana Valenzuela,1,38,58,Bad Behaviour,Substitute
15,3825799,25859,Sérgio Paulo Barbosa Valente,2,62,59,Bad Behaviour,Substitute
22,3825804,27457,Manuel Fernández Muñíz,2,90,47,Bad Behaviour,Substitute


In [ ]:
# ============================================================
# THIS REMOVES NON-PARTICIPATING PLAYER-MATCH ROWS
# ============================================================

historical_corrected = (
    historical_with_corrected_minutes[
        historical_with_corrected_minutes[
            "minutes_played"
        ].notna()
    ]
    .copy()
)

print(
    "Before removal:",
    len(historical_with_corrected_minutes)
)

print(
    "Non-playing Bad Behaviour rows removed:",
    historical_with_corrected_minutes[
        "minutes_played"
    ].isna().sum()
)

print(
    "After removal:",
    len(historical_corrected)
)

print(
    "Matches:",
    historical_corrected[
        "match_id"
    ].nunique()
)

print(
    "Players:",
    historical_corrected[
        "player_id"
    ].nunique()
)

print(
    "Missing minutes:",
    historical_corrected[
        "minutes_played"
    ].isna().sum()
)

print(
    "Duplicate match-player rows:",
    historical_corrected.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

Before removal: 113470
Non-playing Bad Behaviour rows removed: 37
After removal: 113433
Matches: 3961
Players: 9878
Missing minutes: 0
Duplicate match-player rows: 0


In [10]:
# ============================================================
# REGENERATE CORRECTED PER-90 FEATURES
# ============================================================

per90_metrics = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for metric in per90_metrics:

    historical_corrected[
        f"{metric}_per90"
    ] = (
        historical_corrected[metric]
        /
        historical_corrected["minutes_played"]
        * 90
    )


per90_columns = [
    f"{metric}_per90"
    for metric in per90_metrics
]


print(
    historical_corrected[
        per90_columns
    ].describe()
)

       pass_attempts_per90  progressive_passes_per90  carries_per90  \
count        113433.000000             113433.000000  113433.000000   
mean             40.197660                  8.565244      32.333179   
std              21.037619                  6.619213      18.561285   
min               0.000000                  0.000000       0.000000   
25%              25.317604                  3.842505      19.373777   
50%              36.467532                  7.355696      29.189189   
75%              51.244405                 11.848601      41.593625   
max             298.892989                270.000000     323.076923   

         shots_per90  miscontrols_per90  shot_assists_per90  
count  113433.000000      113433.000000       113433.000000  
mean        1.217047           1.450146            0.727661  
std         2.343323           2.295806            1.563203  
min         0.000000           0.000000            0.000000  
25%         0.000000           0.000000           

In [11]:
import numpy as np

print(
    "Infinite per-90 values:",
    np.isinf(
        historical_corrected[
            per90_columns
        ].to_numpy()
    ).sum()
)

print(
    "Missing per-90 values:",
    historical_corrected[
        per90_columns
    ].isna().sum().sum()
)

print(
    "Minimum minutes:",
    historical_corrected[
        "minutes_played"
    ].min()
)

Infinite per-90 values: 0
Missing per-90 values: 0
Minimum minutes: 0.26666666666666666


In [12]:
# ============================================================
# CORRECTED RELIABLE EXPOSURE FLAGS
# ============================================================

historical_corrected[
    "reliable_10min"
] = (
    historical_corrected[
        "minutes_played"
    ] >= 10
)

historical_corrected[
    "reliable_per90"
] = (
    historical_corrected[
        "minutes_played"
    ] >= 20
)

historical_corrected[
    "reliable_30min"
] = (
    historical_corrected[
        "minutes_played"
    ] >= 30
)


for column in [
    "reliable_10min",
    "reliable_per90",
    "reliable_30min"
]:

    count = historical_corrected[column].sum()

    percentage = (
        historical_corrected[column].mean()
        * 100
    )

    print(
        f"{column}: "
        f"{count:,} / "
        f"{len(historical_corrected):,} "
        f"({percentage:.2f}%)"
    )

reliable_10min: 109,938 / 113,433 (96.92%)
reliable_per90: 102,674 / 113,433 (90.52%)
reliable_30min: 95,634 / 113,433 (84.31%)


In [ ]:

for metric in per90_metrics:

    per90_col = f"{metric}_per90"
    reliable_col = f"reliable_{metric}_per90"

    historical_corrected[
        reliable_col
    ] = historical_corrected[
        per90_col
    ].where(
        historical_corrected[
            "reliable_per90"
        ]
    )


reliable_per90_columns = [
    f"reliable_{metric}_per90"
    for metric in per90_metrics
]


print(
    historical_corrected[
        reliable_per90_columns
    ].count()
)

print(
    "\nReliable appearances:",
    historical_corrected[
        "reliable_per90"
    ].sum()
)

print(
    "Unreliable appearances:",
    (
        ~historical_corrected[
            "reliable_per90"
        ]
    ).sum()
)

reliable_pass_attempts_per90         102674
reliable_progressive_passes_per90    102674
reliable_carries_per90               102674
reliable_shots_per90                 102674
reliable_miscontrols_per90           102674
reliable_shot_assists_per90          102674
dtype: int64

Reliable appearances: 102674
Unreliable appearances: 10759


In [14]:


# Make sure observations are in chronological order
historical_corrected = historical_corrected.sort_values(
    ["player_id", "match_date", "match_id"]
).reset_index(drop=True)


# Metrics used for reliable historical baselines
reliable_metrics = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]


# ------------------------------------------------------------
# 1. Previous reliable appearance
# ------------------------------------------------------------

for metric in reliable_metrics:

    source_col = f"reliable_{metric}_per90"

    historical_corrected[
        f"previous_reliable_{metric}_per90"
    ] = (
        historical_corrected
        .groupby("player_id")[source_col]
        .transform(
            lambda s: s.ffill().shift(1)
        )
    )


# ------------------------------------------------------------
# 2. Last 3 reliable appearances
# ------------------------------------------------------------

for metric in reliable_metrics:

    source_col = f"reliable_{metric}_per90"

    historical_corrected[
        f"last_3_reliable_{metric}_per90"
    ] = (
        historical_corrected
        .groupby("player_id")[source_col]
        .transform(
            lambda s:
            s.shift(1)
             .rolling(
                 window=3,
                 min_periods=1
             )
             .mean()
        )
    )


# ------------------------------------------------------------
# 3. Last 5 reliable appearances
# ------------------------------------------------------------

for metric in reliable_metrics:

    source_col = f"reliable_{metric}_per90"

    historical_corrected[
        f"last_5_reliable_{metric}_per90"
    ] = (
        historical_corrected
        .groupby("player_id")[source_col]
        .transform(
            lambda s:
            s.shift(1)
             .rolling(
                 window=5,
                 min_periods=1
             )
             .mean()
        )
    )


print("Shape:", historical_corrected.shape)

print(
    "\nPrevious reliable pass-attempt baseline available:",
    historical_corrected[
        "previous_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Last-3 reliable pass-attempt baseline available:",
    historical_corrected[
        "last_3_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Last-5 reliable pass-attempt baseline available:",
    historical_corrected[
        "last_5_reliable_pass_attempts_per90"
    ].notna().sum()
)

Shape: (113433, 83)

Previous reliable pass-attempt baseline available: 101836
Last-3 reliable pass-attempt baseline available: 101202
Last-5 reliable pass-attempt baseline available: 101746


In [15]:
# ============================================================
# CORRECTED BASELINES OVER RELIABLE APPEARANCES ONLY
# ============================================================

historical_corrected = historical_corrected.sort_values(
    ["player_id", "match_date", "match_id"]
).reset_index(drop=True)

reliable_metrics = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]


def calculate_reliable_history(player_df):

    player_df = player_df.copy()

    # Store previous reliable observations encountered so far
    history = {
        metric: []
        for metric in reliable_metrics
    }

    previous_values = {
        metric: []
        for metric in reliable_metrics
    }

    last3_values = {
        metric: []
        for metric in reliable_metrics
    }

    last5_values = {
        metric: []
        for metric in reliable_metrics
    }

    for _, row in player_df.iterrows():

        # ----------------------------------------------------
        # Calculate history BEFORE adding current match
        # This prevents current-match leakage
        # ----------------------------------------------------

        for metric in reliable_metrics:

            values = history[metric]

            previous_values[metric].append(
                values[-1]
                if len(values) >= 1
                else np.nan
            )

            last3_values[metric].append(
                np.mean(values[-3:])
                if len(values) >= 1
                else np.nan
            )

            last5_values[metric].append(
                np.mean(values[-5:])
                if len(values) >= 1
                else np.nan
            )

        # ----------------------------------------------------
        # Only AFTER baseline calculation do we add the
        # current appearance, and only if it is reliable
        # ----------------------------------------------------

        if row["reliable_per90"]:

            for metric in reliable_metrics:

                history[metric].append(
                    row[
                        f"reliable_{metric}_per90"
                    ]
                )


    for metric in reliable_metrics:

        player_df[
            f"previous_reliable_{metric}_per90"
        ] = previous_values[metric]

        player_df[
            f"last_3_reliable_{metric}_per90"
        ] = last3_values[metric]

        player_df[
            f"last_5_reliable_{metric}_per90"
        ] = last5_values[metric]


    return player_df


historical_corrected = (
    historical_corrected
    .groupby(
        "player_id",
        group_keys=False
    )
    .apply(
        calculate_reliable_history,
        include_groups=False
    )
    .reset_index()
)


print(
    "Previous reliable available:",
    historical_corrected[
        "previous_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Last-3 reliable available:",
    historical_corrected[
        "last_3_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Last-5 reliable available:",
    historical_corrected[
        "last_5_reliable_pass_attempts_per90"
    ].notna().sum()
)

Previous reliable available: 101836
Last-3 reliable available: 101836
Last-5 reliable available: 101836


In [16]:
# ============================================================
# MANUAL VALIDATION OF LEAKAGE-SAFE HISTORY
# ============================================================

player_check = historical_corrected[
    historical_corrected[
        "player_name"
    ].str.contains(
        "Marc-André ter Stegen",
        case=False,
        na=False
    )
].copy()

player_check = player_check.sort_values(
    ["match_date", "match_id"]
)

display(
    player_check[
        [
            "match_date",
            "match_id",
            "minutes_played",
            "pass_attempts",
            "pass_attempts_per90",
            "reliable_per90",
            "previous_reliable_pass_attempts_per90",
            "last_3_reliable_pass_attempts_per90",
            "last_5_reliable_pass_attempts_per90"
        ]
    ].head(12)
)

,match_date,match_id,minutes_played,pass_attempts,pass_attempts_per90,reliable_per90,previous_reliable_pass_attempts_per90,last_3_reliable_pass_attempts_per90,last_5_reliable_pass_attempts_per90
67179,2015-06-06,18242,96.983333,29,26.911841,True,NaN,NaN,NaN
67180,2015-09-12,266166,93.066667,34,32.879656,True,26.911841,26.911841,26.911841
67181,2015-09-20,266490,92.566667,33,32.084984,True,32.879656,29.895748,29.895748
67182,2015-09-23,266467,92.050000,33,32.265073,True,32.084984,30.625493,30.625493
67183,2015-09-26,267611,95.450000,31,29.229963,True,32.265073,32.409904,31.035388
67184,2016-04-30,266986,4.983333,1,18.060201,False,29.229963,31.193340,30.674303
67185,2016-05-08,265958,92.650000,23,22.342148,True,29.229963,31.193340,30.674303
67186,2016-05-14,267506,94.250000,32,30.557029,True,22.342148,27.945728,29.760365
67187,2016-08-28,266892,93.916667,73,69.955634,True,30.557029,27.376380,29.295840
67188,2016-09-17,267212,92.366667,56,54.565139,True,69.955634,40.951604,36.869970


In [18]:
print(historical_corrected.columns.tolist())

print("\nIndex names:")
print(historical_corrected.index.names)

print("\nShape:")
print(historical_corrected.shape)

['index', 'match_id', 'player_name', 'total_events', 'pass_attempts', 'passes_completed', 'forward_passes', 'progressive_passes', 'avg_pass_length', 'avg_pass_angle_degrees', 'crosses', 'through_balls', 'switches', 'shot_assists', 'goal_assists', 'shots', 'carries', 'miscontrols', 'pass_completion_rate', 'match_date', 'home_team_name', 'away_team_name', 'prev_pass_completion_rate', 'rolling_3_pass_completion', 'rolling_5_pass_completion', 'prev_pass_attempts', 'rolling_3_pass_attempts', 'rolling_5_pass_attempts', 'prev_progressive_passes', 'rolling_3_progressive_passes', 'rolling_5_progressive_passes', 'prev_carries', 'rolling_3_carries', 'rolling_5_carries', 'prev_shots', 'rolling_3_shots', 'rolling_5_shots', 'prev_miscontrols', 'rolling_3_miscontrols', 'rolling_5_miscontrols', 'prev_shot_assists', 'rolling_3_shot_assists', 'rolling_5_shot_assists', 'previous_matches_available', 'minutes_played', 'first_half_minutes', 'second_half_minutes', 'extra_time_minutes', 'temporary_off_pitch_g

In [19]:
# ============================================================
# REBUILD RELIABLE HISTORICAL BASELINES
# WHILE PRESERVING PLAYER_ID
# ============================================================

historical_corrected = historical_corrected.reset_index(drop=False)

# If player_id was stored in the old index column,
# restore it explicitly
if "player_id" not in historical_corrected.columns:

    possible_id_columns = [
        col for col in historical_corrected.columns
        if col.startswith("level_")
    ]

    if len(possible_id_columns) > 0:
        historical_corrected = historical_corrected.rename(
            columns={
                possible_id_columns[0]: "player_id"
            }
        )


print(
    "player_id restored:",
    "player_id" in historical_corrected.columns
)

print(
    "Shape:",
    historical_corrected.shape
)

player_id restored: True
Shape: (113433, 84)


In [20]:
# ============================================================
# FINAL STRUCTURE CHECK BEFORE SAVING
# ============================================================

print("Shape:", historical_corrected.shape)

print(
    "player_id present:",
    "player_id" in historical_corrected.columns
)

print(
    "Missing player IDs:",
    historical_corrected["player_id"].isna().sum()
)

print(
    "Unique players:",
    historical_corrected["player_id"].nunique()
)

print(
    "Duplicate match-player rows:",
    historical_corrected.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

print("\nFirst 5 columns:")
print(historical_corrected.columns[:5].tolist())

Shape: (113433, 84)
player_id present: True
Missing player IDs: 0
Unique players: 113433
Duplicate match-player rows: 0

First 5 columns:
['player_id', 'index', 'match_id', 'player_name', 'total_events']


In [21]:
# ============================================================
# CLEAN REBUILD OF CORRECTED HISTORICAL FEATURES
# PRESERVES TRUE PLAYER_ID
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Start again from the correctly merged historical dataset
# ------------------------------------------------------------

historical_corrected = (
    historical_with_corrected_minutes[
        historical_with_corrected_minutes[
            "minutes_played"
        ].notna()
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Rebuild six per-90 features
# ------------------------------------------------------------

per90_metrics = [
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for metric in per90_metrics:

    historical_corrected[
        f"{metric}_per90"
    ] = (
        historical_corrected[metric]
        /
        historical_corrected["minutes_played"]
        * 90
    )


# ------------------------------------------------------------
# 3. Rebuild exposure reliability flags
# ------------------------------------------------------------

historical_corrected[
    "reliable_10min"
] = historical_corrected[
    "minutes_played"
] >= 10

historical_corrected[
    "reliable_per90"
] = historical_corrected[
    "minutes_played"
] >= 20

historical_corrected[
    "reliable_30min"
] = historical_corrected[
    "minutes_played"
] >= 30


# ------------------------------------------------------------
# 4. Create reliable per-90 source columns
# ------------------------------------------------------------

for metric in per90_metrics:

    historical_corrected[
        f"reliable_{metric}_per90"
    ] = historical_corrected[
        f"{metric}_per90"
    ].where(
        historical_corrected[
            "reliable_per90"
        ]
    )


# ------------------------------------------------------------
# 5. Sort chronologically
# ------------------------------------------------------------

historical_corrected = (
    historical_corrected
    .sort_values(
        [
            "player_id",
            "match_date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Prepare leakage-safe baseline columns
# ------------------------------------------------------------

for metric in per90_metrics:

    historical_corrected[
        f"previous_reliable_{metric}_per90"
    ] = np.nan

    historical_corrected[
        f"last_3_reliable_{metric}_per90"
    ] = np.nan

    historical_corrected[
        f"last_5_reliable_{metric}_per90"
    ] = np.nan


# ------------------------------------------------------------
# 7. Calculate history using ONLY prior reliable appearances
# ------------------------------------------------------------

for player_id, indices in historical_corrected.groupby(
    "player_id",
    sort=False
).groups.items():

    history = {
        metric: []
        for metric in per90_metrics
    }

    for idx in indices:

        # Calculate baselines BEFORE adding current match
        for metric in per90_metrics:

            values = history[metric]

            if len(values) > 0:

                historical_corrected.at[
                    idx,
                    f"previous_reliable_{metric}_per90"
                ] = values[-1]

                historical_corrected.at[
                    idx,
                    f"last_3_reliable_{metric}_per90"
                ] = np.mean(
                    values[-3:]
                )

                historical_corrected.at[
                    idx,
                    f"last_5_reliable_{metric}_per90"
                ] = np.mean(
                    values[-5:]
                )

        # Add current match only AFTER baseline calculation
        if historical_corrected.at[
            idx,
            "reliable_per90"
        ]:

            for metric in per90_metrics:

                history[metric].append(
                    historical_corrected.at[
                        idx,
                        f"reliable_{metric}_per90"
                    ]
                )


# ------------------------------------------------------------
# 8. Final structural validation
# ------------------------------------------------------------

print(
    "Shape:",
    historical_corrected.shape
)

print(
    "Unique players:",
    historical_corrected[
        "player_id"
    ].nunique()
)

print(
    "Missing player IDs:",
    historical_corrected[
        "player_id"
    ].isna().sum()
)

print(
    "Duplicate match-player rows:",
    historical_corrected.duplicated(
        subset=[
            "match_id",
            "player_id"
        ]
    ).sum()
)

print(
    "Reliable appearances:",
    historical_corrected[
        "reliable_per90"
    ].sum()
)

print(
    "Previous reliable baseline available:",
    historical_corrected[
        "previous_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Last-3 baseline available:",
    historical_corrected[
        "last_3_reliable_pass_attempts_per90"
    ].notna().sum()
)

print(
    "Last-5 baseline available:",
    historical_corrected[
        "last_5_reliable_pass_attempts_per90"
    ].notna().sum()
)

Shape: (113433, 83)
Unique players: 9878
Missing player IDs: 0
Duplicate match-player rows: 0
Reliable appearances: 102674
Previous reliable baseline available: 101836
Last-3 baseline available: 101836
Last-5 baseline available: 101836


In [22]:
# ============================================================
# SAVE FINAL CORRECTED HISTORICAL FEATURE DATASET
# ============================================================

historical_corrected.to_pickle(
    "historical_player_match_master_corrected.pkl"
)

print(
    "Saved: historical_player_match_master_corrected.pkl"
)

print(
    "Shape:",
    historical_corrected.shape
)

print(
    "Matches:",
    historical_corrected["match_id"].nunique()
)

print(
    "Players:",
    historical_corrected["player_id"].nunique()
)

print(
    "Reliable appearances:",
    historical_corrected["reliable_per90"].sum()
)

print(
    "Duplicate match-player rows:",
    historical_corrected.duplicated(
        subset=["match_id", "player_id"]
    ).sum()
)

Saved: historical_player_match_master_corrected.pkl
Shape: (113433, 83)
Matches: 3961
Players: 9878
Reliable appearances: 102674
Duplicate match-player rows: 0
